In [2]:
# ============================================================
# CELL 1 — Benchmark data partition and provenance
# ============================================================
# Establish the approved chronological development and
# evaluation partitions without modifying original data.

from pathlib import Path
from datetime import datetime
import hashlib
import csv
import json
import sys

# Locate the project and verify the Python environment.
ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "PROTOCOL.md").is_file()
)

DATA = ROOT / "data"
RAW = DATA / "raw"

assert Path(sys.prefix).resolve() == (
    ROOT / ".venv"
).resolve()

SOURCE_COMMIT = (DATA / "source_commit.txt").read_text().strip()

# Verify the integrity of the original source files.
manifest = {}

for line in (DATA / "SHA256SUMS").read_text().splitlines():
    digest, filename = line.split(maxsplit=1)
    manifest[filename] = digest

actual_files = {
    path.relative_to(ROOT).as_posix()
    for path in RAW.rglob("*")
    if path.is_file()
}

assert actual_files == set(manifest)

for filename, expected_hash in manifest.items():
    assert hashlib.sha256(
        (ROOT / filename).read_bytes()
    ).hexdigest() == expected_hash

# Load the original batch definitions.
proofs = json.loads(
    (RAW / "Meta Data/proof_log.json").read_text()
)

# Preserve the original CSV records and row positions.
records_by_file = {}

for file in sorted((RAW / "logs").glob("*.csv")):
    with file.open(encoding="utf-8", newline="") as stream:
        records_by_file[file.name] = list(csv.DictReader(stream))

# Approved experimental allocation.
# Source: EXPERIMENTAL_DESIGN.md, Section 2.
DEVELOPMENT_COUNT = 65
EVALUATION_COUNT = 65

assert len(proofs) == DEVELOPMENT_COUNT + EVALUATION_COUNT

# Reconstruct and audit the chronological partitions.
development_batches = []
evaluation_batch_indices = []

development_hashes = set()
evaluation_hashes = set()

partition_times = {
    "development": [],
    "evaluation": [],
}

previous_end = None

for index, proof in enumerate(proofs):

    filename = Path(proof["csv_path"]).name
    start = proof["row_start"]
    end = proof["row_end"]

    assert 0 <= start <= end < len(records_by_file[filename])

    records = records_by_file[filename][start:end + 1]

    assert len(records) == proof["size"]

    timestamps = [
        datetime.fromisoformat(record["timestamp"])
        for record in records
    ]

    assert all(
        current > previous
        for previous, current in zip(
            timestamps, timestamps[1:]
        )
    )

    first_time = timestamps[0]
    last_time = timestamps[-1]

    if previous_end is not None:
        assert first_time > previous_end

    previous_end = last_time

    hashes = {
        record["chain_hash"] for record in records
    }

    assert len(hashes) == len(records)

    if index < DEVELOPMENT_COUNT:

        assert not development_hashes.intersection(hashes)
        development_hashes.update(hashes)

        development_batches.append({
            "batch_index": index,
            "proof": proof,
            "records": records,
        })

        partition_times["development"].extend(
            [first_time, last_time]
        )

    else:

        assert not evaluation_hashes.intersection(hashes)
        evaluation_hashes.update(hashes)

        evaluation_batch_indices.append(index)

        partition_times["evaluation"].extend(
            [first_time, last_time]
        )

# Verify separation between the two partitions.
overlap = development_hashes & evaluation_hashes

assert len(development_batches) == DEVELOPMENT_COUNT
assert len(evaluation_batch_indices) == EVALUATION_COUNT
assert not overlap

assert (
    max(partition_times["development"])
    < min(partition_times["evaluation"])
)

print("=== DATA PROVENANCE ===")
print("Source commit:", SOURCE_COMMIT)
print("Verified original files:", len(manifest))

print("\n=== DEVELOPMENT PARTITION ===")
print("Batches:", len(development_batches))
print("Unique measurements:", len(development_hashes))
print("First timestamp:", min(partition_times["development"]))
print("Last timestamp:", max(partition_times["development"]))

print("\n=== EVALUATION PARTITION ===")
print("Batches:", len(evaluation_batch_indices))
print("Unique measurements:", len(evaluation_hashes))
print("First timestamp:", min(partition_times["evaluation"]))
print("Last timestamp:", max(partition_times["evaluation"]))

print("\n=== PARTITION VALIDATION ===")
print("Overlapping records:", len(overlap))
print("Chronological separation: PASS")
print("Total referenced measurements:",
      len(development_hashes | evaluation_hashes))

# Retain development payloads only.
# Evaluation data remain available in the original source files.
del records_by_file, proofs, records, proof, timestamps, hashes

print("\nVALIDATION: PASS")

=== DATA PROVENANCE ===
Source commit: ec34ea5dc21642744d39eb436b787ac092505d0f
Verified original files: 12

=== DEVELOPMENT PARTITION ===
Batches: 65
Unique measurements: 3874
First timestamp: 2025-06-12 18:32:12.452010
Last timestamp: 2025-06-15 11:05:12.972425

=== EVALUATION PARTITION ===
Batches: 65
Unique measurements: 4211
First timestamp: 2025-06-15 11:06:12.972963
Last timestamp: 2025-06-18 09:16:13.539235

=== PARTITION VALIDATION ===
Overlapping records: 0
Chronological separation: PASS
Total referenced measurements: 8085

VALIDATION: PASS


In [3]:
# ============================================================
# CELL 2 — Canonical evidence representation (Version 2)
# ============================================================
# Construct equivalent signed evidence for all architectures.
#
# Approved refinement: PROTOCOL_AMENDMENT_02.md
# Serialization reference: RFC 8785, Sections 3.1 and 3.2.
# https://doi.org/10.17487/RFC8785
#
# Measurement values and original record order are preserved.

from copy import deepcopy
from datetime import timezone

SCHEMA_VERSION = "iot-evidence-benchmark-v2"

# Units documented in the original logger implementation.
# These declarations become part of the signed evidence.
MEASUREMENT_UNITS = {
    "voltage": "V",
    "current": "A",
    "power": "W",
    "energy_Wh": "Wh",
    "cumulative_energy_Wh": "Wh",
}


def validate_json_types(value):
    """Restrict evidence to deterministic JSON-compatible types."""

    if type(value) is dict:

        for key, item in value.items():

            if type(key) is not str:
                raise TypeError("JSON keys must be strings.")

            validate_json_types(item)

    elif type(value) is list:

        for item in value:
            validate_json_types(item)

    elif type(value) not in (str, int, bool, type(None)):

        raise TypeError(
            f"Unsupported evidence type: {type(value).__name__}"
        )


def canonical_bytes(evidence):
    """Serialize evidence using fixed benchmark JSON rules."""

    validate_json_types(evidence)

    return json.dumps(
        evidence,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")


def evidence_commitment(evidence):
    """Calculate the SHA-256 commitment."""

    return hashlib.sha256(
        canonical_bytes(evidence)
    ).hexdigest()


def build_evidence(batch, previous_commitment):
    """Construct a benchmark evidence object from original records."""

    records = deepcopy(batch["records"])

    assert records

    source_ids = {
        record["device_id"] for record in records
    }

    assert len(source_ids) == 1

    # Synthetic control timestamp, not historical event time.
    application_time = datetime.fromisoformat(
        records[-1]["timestamp"]
    ).replace(tzinfo=timezone.utc)

    return {
        "schema": SCHEMA_VERSION,
        "sequence": batch["batch_index"],
        "source_id": next(iter(source_ids)),
        "previous_commitment": previous_commitment,
        "claims": {
            "application_timestamp_utc": (
                application_time.isoformat()
            ),
            "metadata_origin": "synthetic_control",
            "measurement_units": deepcopy(MEASUREMENT_UNITS),
        },
        "records": records,
    }


# ------------------------------------------------------------
# Construct development evidence only
# ------------------------------------------------------------

development_evidence = []

previous_commitment = None

for batch in development_batches:

    evidence = build_evidence(
        batch, previous_commitment
    )

    commitment = evidence_commitment(evidence)

    development_evidence.append({
        "evidence": evidence,
        "commitment": commitment,
    })

    previous_commitment = commitment


# ------------------------------------------------------------
# Implementation validation
# ------------------------------------------------------------

# Test 1: Fixed serialization example.
assert canonical_bytes({"b": 2, "a": 1}) == b'{"a":1,"b":2}'

# Test 2: Dictionary order invariance.
assert canonical_bytes({"a": 1, "b": 2}) == (
    canonical_bytes({"b": 2, "a": 1})
)

# Test 3: Floating-point restriction.
try:
    canonical_bytes({"measurement": 0.1})
except TypeError:
    pass
else:
    raise AssertionError("Floating-point restriction failed.")

# Test 4: Original record preservation.
assert all(
    item["evidence"]["records"] == batch["records"]
    for item, batch in zip(
        development_evidence,
        development_batches,
        strict=True,
    )
)

# Test 5: Commitment chain consistency.
expected_previous = None

for item in development_evidence:

    evidence = item["evidence"]

    assert evidence["previous_commitment"] == expected_previous

    assert evidence_commitment(evidence) == item["commitment"]

    expected_previous = item["commitment"]

# Test 6: Metadata modification detection.
original = development_evidence[0]["evidence"]

modified = deepcopy(original)
modified["claims"]["application_timestamp_utc"] += "_modified"

assert evidence_commitment(original) != (
    evidence_commitment(modified)
)

# Test 7: Record order sensitivity.
reordered = deepcopy(original)
reordered["records"].reverse()

assert evidence_commitment(original) != (
    evidence_commitment(reordered)
)

# Test 8: Unit declaration preservation.
assert all(
    item["evidence"]["claims"]["measurement_units"]
    == MEASUREMENT_UNITS
    for item in development_evidence
)

# Test 9: Unit modification changes the commitment.
modified_units = deepcopy(original)
modified_units["claims"]["measurement_units"]["current"] = "mA"

assert evidence_commitment(original) != (
    evidence_commitment(modified_units)
)

# Verify that modifying a copied object does not alter
# the original evidence or construction configuration.
assert original["claims"]["measurement_units"]["current"] == "A"
assert MEASUREMENT_UNITS["current"] == "A"

# Test 10: Deterministic evidence reconstruction.
assert build_evidence(
    development_batches[0], None
) == original

commitments = {
    item["commitment"]
    for item in development_evidence
}

assert len(commitments) == len(development_evidence)

# Verify that version 2 differs from the archived version.
ARCHIVED_V1_FIRST_COMMITMENT = (
    "33d5d385357f2975ba23e986c5d40fa41"
    "ac0b009bc203d210033e224132e42f5"
)

assert development_evidence[0]["commitment"] != (
    ARCHIVED_V1_FIRST_COMMITMENT
)


# ------------------------------------------------------------
# Report validation results
# ------------------------------------------------------------

print("=== CANONICAL EVIDENCE ===")
print("Schema:", SCHEMA_VERSION)
print("Development evidence objects:", len(development_evidence))
print("Unique commitments:", len(commitments))

print("Measurement records:", sum(
    len(item["evidence"]["records"])
    for item in development_evidence
))

print("\n=== SIGNED UNIT DECLARATIONS ===")

for field, unit in MEASUREMENT_UNITS.items():
    print(f"{field}: {unit}")

print("\n=== IMPLEMENTATION TESTS ===")
print("Reference serialization: PASS")
print("Dictionary order invariance: PASS")
print("Floating-point restriction: PASS")
print("Original record preservation: PASS")
print("Commitment chain consistency: PASS")
print("Metadata modification detection: PASS")
print("Record order sensitivity: PASS")
print("Unit declaration preservation: PASS")
print("Unit modification detection: PASS")
print("Deterministic reconstruction: PASS")

print("\n=== COMMITMENT BOUNDARIES ===")
print("First commitment:", development_evidence[0]["commitment"])
print("Last commitment:", development_evidence[-1]["commitment"])
print("Change from archived version 1: CONFIRMED")

print("\nVALIDATION: PASS")

=== CANONICAL EVIDENCE ===
Schema: iot-evidence-benchmark-v2
Development evidence objects: 65
Unique commitments: 65
Measurement records: 3874

=== SIGNED UNIT DECLARATIONS ===
voltage: V
current: A
power: W
energy_Wh: Wh
cumulative_energy_Wh: Wh

=== IMPLEMENTATION TESTS ===
Reference serialization: PASS
Dictionary order invariance: PASS
Floating-point restriction: PASS
Original record preservation: PASS
Commitment chain consistency: PASS
Metadata modification detection: PASS
Record order sensitivity: PASS
Unit declaration preservation: PASS
Unit modification detection: PASS
Deterministic reconstruction: PASS

=== COMMITMENT BOUNDARIES ===
First commitment: e8041dc76cfffab806f34939fffe2ab5faeec6dfb5a5920a00685548805fff93
Last commitment: b95e58c148f97265dfef56a9fd718d3e38e357a458d629112d443a6f1638f2ec
Change from archived version 1: CONFIRMED

VALIDATION: PASS


In [4]:
# ============================================================
# CELL 3 — Common digital signature mechanism
# ============================================================
# Implement Ed25519 signing, verification and authorization.
#
# Reference: RFC 8032, Sections 5.1 and 7.1.
# https://doi.org/10.17487/RFC8032
#
# WARNING:
# Deterministic keys are public experimental fixtures.
# They must never be used in production systems.

from Crypto.Signature import eddsa


SIGNATURE_DOMAIN = (
    f"{SCHEMA_VERSION}:source-signature".encode("utf-8")
    + b"\x00"
)

def experimental_key(role):
    """Generate a reproducible experimental Ed25519 key."""

    seed = hashlib.sha256(
        f"{SCHEMA_VERSION}:experimental-key:{role}".encode("utf-8")
    ).digest()

    return eddsa.import_private_key(seed)


def public_key_bytes(private_key):
    """Return the raw Ed25519 public key."""

    return private_key.public_key().export_key(format="raw")


def sign_evidence(evidence, private_key):
    """Sign the canonical evidence representation."""

    message = SIGNATURE_DOMAIN + canonical_bytes(evidence)

    return eddsa.new(
        private_key, "rfc8032"
    ).sign(message)


def verify_signature(evidence, signature, public_key):
    """Verify cryptographic authenticity independently of authorization."""

    if len(public_key) != 32 or len(signature) != 64:
        return False

    try:
        key = eddsa.import_public_key(public_key)

        message = SIGNATURE_DOMAIN + canonical_bytes(evidence)

        eddsa.new(key, "rfc8032").verify(
            message, signature
        )

        return True

    except (ValueError, TypeError):
        return False


# ------------------------------------------------------------
# Independent RFC 8032 reference test
# ------------------------------------------------------------

reference_seed = bytes.fromhex(
    "9d61b19deffd5a60ba844af492ec2cc4"
    "4449c5697b326919703bac031cae7f60"
)

reference_public_key = bytes.fromhex(
    "d75a980182b10ab7d54bfed3c964073a"
    "0ee172f3daa62325af021a68f707511a"
)

reference_signature = bytes.fromhex(
    "e5564300c360ac729086e2cc806e828a"
    "84877f1eb8e5d974d873e06522490155"
    "5fb8821590a33bacc61e39701cf9b46b"
    "d25bf5f0595bbe24655141438e7a100b"
)

reference_key = eddsa.import_private_key(reference_seed)

assert public_key_bytes(reference_key) == reference_public_key

assert eddsa.new(
    reference_key, "rfc8032"
).sign(b"") == reference_signature

eddsa.new(
    eddsa.import_public_key(reference_public_key),
    "rfc8032"
).verify(b"", reference_signature)


# ------------------------------------------------------------
# Experimental identities and authorization registry
# ------------------------------------------------------------

source_ids = {
    item["evidence"]["source_id"]
    for item in development_evidence
}

source_private_keys = {
    source_id: experimental_key(f"source:{source_id}")
    for source_id in sorted(source_ids)
}

# The registry is configured by the experiment,
# not supplied by the evidence issuer.
authorized_sources = {
    source_id: public_key_bytes(key)
    for source_id, key in source_private_keys.items()
}

unauthorized_key = experimental_key("unauthorized-source")


def is_authorized(evidence, public_key):
    """Check the configured source authorization policy."""

    return authorized_sources.get(
        evidence["source_id"]
    ) == public_key


# ------------------------------------------------------------
# Sign and verify all development evidence
# ------------------------------------------------------------

signed_development_evidence = []

for item in development_evidence:

    evidence = item["evidence"]
    source_id = evidence["source_id"]

    private_key = source_private_keys[source_id]
    public_key = authorized_sources[source_id]

    signature = sign_evidence(evidence, private_key)

    assert verify_signature(
        evidence, signature, public_key
    )

    assert is_authorized(evidence, public_key)

    signed_development_evidence.append({
        "evidence": deepcopy(evidence),
        "commitment": item["commitment"],
        "public_key": public_key,
        "signature": signature,
    })


# ------------------------------------------------------------
# Controlled positive and negative tests
# ------------------------------------------------------------

sample = signed_development_evidence[0]

original = sample["evidence"]
signature = sample["signature"]
public_key = sample["public_key"]

# Test 1: Valid signature.
assert verify_signature(original, signature, public_key)

# Test 2: Modified evidence.
modified = deepcopy(original)
modified["claims"]["application_timestamp_utc"] += "_modified"

assert not verify_signature(modified, signature, public_key)

# Test 3: Modified signature.
modified_signature = bytes([signature[0] ^ 1]) + signature[1:]

assert not verify_signature(
    original, modified_signature, public_key
)

# Test 4: Incorrect verification key.
other_public_key = public_key_bytes(unauthorized_key)

assert not verify_signature(
    original, signature, other_public_key
)

# Test 5: Valid signature from an unauthorized issuer.
other_signature = sign_evidence(original, unauthorized_key)

assert verify_signature(
    original, other_signature, other_public_key
)

assert not is_authorized(original, other_public_key)

# Test 6: Malformed signature.
assert not verify_signature(
    original, signature[:-1], public_key
)

# Test 7: Deterministic signing.
assert sign_evidence(
    original,
    source_private_keys[original["source_id"]]
) == signature


# ------------------------------------------------------------
# Report validation results
# ------------------------------------------------------------

print("=== DIGITAL SIGNATURE CONFIGURATION ===")
print("Algorithm: Ed25519")
print("Authorized sources:", len(authorized_sources))
print("Public key length:", len(public_key), "bytes")
print("Signature length:", len(signature), "bytes")

print("\n=== REFERENCE VALIDATION ===")
print("RFC 8032 public key reproduction: PASS")
print("RFC 8032 signature reproduction: PASS")
print("RFC 8032 signature verification: PASS")

print("\n=== DEVELOPMENT EVIDENCE ===")
print("Signed batches:", len(signed_development_evidence))
print("Valid signatures:", len(signed_development_evidence))
print("Authorized signatures:", len(signed_development_evidence))

print("\n=== CONTROLLED TESTS ===")
print("Valid signature: PASS")
print("Modified evidence detection: PASS")
print("Modified signature detection: PASS")
print("Incorrect key detection: PASS")
print("Unauthorized issuer detection: PASS")
print("Malformed signature detection: PASS")
print("Deterministic signing: PASS")

print("\nVALIDATION: PASS")

=== DIGITAL SIGNATURE CONFIGURATION ===
Algorithm: Ed25519
Authorized sources: 1
Public key length: 32 bytes
Signature length: 64 bytes

=== REFERENCE VALIDATION ===
RFC 8032 public key reproduction: PASS
RFC 8032 signature reproduction: PASS
RFC 8032 signature verification: PASS

=== DEVELOPMENT EVIDENCE ===
Signed batches: 65
Valid signatures: 65
Authorized signatures: 65

=== CONTROLLED TESTS ===
Valid signature: PASS
Modified evidence detection: PASS
Modified signature detection: PASS
Incorrect key detection: PASS
Unauthorized issuer detection: PASS
Malformed signature detection: PASS
Deterministic signing: PASS

VALIDATION: PASS


In [5]:
# ============================================================
# CELL 4 — Transparency log and Merkle inclusion proofs
# ============================================================
# Implement the Merkle construction and inclusion verification
# specified in RFC 9162, Sections 2.1.1 and 2.1.3.
#
# Reference: https://doi.org/10.17487/RFC9162
#
# This is a research transparency log, not a complete
# Certificate Transparency protocol implementation.


def log_leaf_hash(entry):
    """Hash a log entry with leaf-domain separation."""

    return hashlib.sha256(b"\x00" + entry).digest()


def log_node_hash(left, right):
    """Hash two child nodes with node-domain separation."""

    return hashlib.sha256(
        b"\x01" + left + right
    ).digest()


def log_split(size):
    """Return the largest power of two smaller than size."""

    return 1 << ((size - 1).bit_length() - 1)


def log_tree_root(entries):
    """Calculate the RFC 9162 Merkle tree root."""

    size = len(entries)

    if size == 0:
        return hashlib.sha256(b"").digest()

    if size == 1:
        return log_leaf_hash(entries[0])

    k = log_split(size)

    return log_node_hash(
        log_tree_root(entries[:k]),
        log_tree_root(entries[k:])
    )


def log_inclusion_proof(entries, index):
    """Generate a Merkle inclusion path for one log entry."""

    size = len(entries)

    if not 0 <= index < size:
        raise ValueError("Invalid leaf index.")

    if size == 1:
        return []

    k = log_split(size)

    if index < k:
        return log_inclusion_proof(
            entries[:k], index
        ) + [log_tree_root(entries[k:])]

    return log_inclusion_proof(
        entries[k:], index - k
    ) + [log_tree_root(entries[:k])]


def verify_log_inclusion(entry, index, size, proof, root):
    """Verify inclusion using RFC 9162 Section 2.1.3.2."""

    if (
        not 0 <= index < size
        or len(root) != 32
        or any(len(node) != 32 for node in proof)
    ):
        return False

    fn = index
    sn = size - 1

    result = log_leaf_hash(entry)

    for node in proof:

        if sn == 0:
            return False

        if fn & 1 or fn == sn:

            result = log_node_hash(node, result)

            while fn and not (fn & 1):
                fn >>= 1
                sn >>= 1

        else:

            result = log_node_hash(result, node)

        fn >>= 1
        sn >>= 1

    return sn == 0 and result == root


# ------------------------------------------------------------
# Reference checks
# ------------------------------------------------------------

# Known SHA-256 digest of an empty byte string.
assert log_tree_root([]).hex() == (
    "e3b0c44298fc1c149afbf4c8996fb924"
    "27ae41e4649b934ca495991b7852b855"
)

# Single-leaf boundary condition.
assert log_inclusion_proof([b"a"], 0) == []

# Reproduce the seven-leaf tree structure illustrated
# in RFC 9162, Section 2.1.5.
example_entries = [
    f"d{i}".encode() for i in range(7)
]

# The seventh leaf requires two sibling subtree hashes.
assert log_inclusion_proof(example_entries, 6) == [
    log_tree_root(example_entries[4:6]),
    log_tree_root(example_entries[:4]),
]


# ------------------------------------------------------------
# Construct the development transparency log
# ------------------------------------------------------------

log_entries = [
    bytes.fromhex(item["commitment"])
    for item in signed_development_evidence
]

assert all(len(entry) == 32 for entry in log_entries)

assert len(set(log_entries)) == len(log_entries)

# Store the root at each chronological tree size.
log_roots = {
    size: log_tree_root(log_entries[:size])
    for size in range(len(log_entries) + 1)
}


# ------------------------------------------------------------
# Exhaustive development inclusion tests
# ------------------------------------------------------------

valid_tests = 0
negative_tests = 0

for size in range(1, len(log_entries) + 1):

    prefix = log_entries[:size]
    root = log_roots[size]

    for index, entry in enumerate(prefix):

        proof = log_inclusion_proof(prefix, index)

        # Positive test: correct inclusion proof.
        assert verify_log_inclusion(
            entry, index, size, proof, root
        )

        valid_tests += 1

        # Negative test: altered entry.
        assert not verify_log_inclusion(
            b"\xff" + entry, index, size, proof, root
        )

        # Negative test: excessive proof length.
        assert not verify_log_inclusion(
            entry, index, size,
            proof + [b"\x00" * 32], root
        )

        # Negative test: invalid leaf index.
        assert not verify_log_inclusion(
            entry, size, size, proof, root
        )

        negative_tests += 3

        # Negative test: incomplete proof.
        if proof:

            assert not verify_log_inclusion(
                entry, index, size, proof[:-1], root
            )

            negative_tests += 1


# ------------------------------------------------------------
# Report results
# ------------------------------------------------------------

print("=== TRANSPARENCY LOG CONFIGURATION ===")
print("Hash algorithm: SHA-256")
print("Log entries:", len(log_entries))
print("Unique log entries:", len(set(log_entries)))
print("Tree checkpoints:", len(log_roots))
print("Leaf and node domain separation: ENABLED")

print("\n=== REFERENCE VALIDATION ===")
print("Empty tree reference: PASS")
print("Single-leaf boundary: PASS")
print("RFC seven-leaf example: PASS")

print("\n=== INCLUSION VERIFICATION ===")
print("Positive tests passed:", valid_tests)
print("Negative tests passed:", negative_tests)
print("Total inclusion tests:", valid_tests + negative_tests)

print("\n=== TREE BOUNDARIES ===")
print("Empty root:", log_roots[0].hex())
print("First root:", log_roots[1].hex())
print("Final root:", log_roots[len(log_entries)].hex())

print("\nVALIDATION: PASS")

=== TRANSPARENCY LOG CONFIGURATION ===
Hash algorithm: SHA-256
Log entries: 65
Unique log entries: 65
Tree checkpoints: 66
Leaf and node domain separation: ENABLED

=== REFERENCE VALIDATION ===
Empty tree reference: PASS
Single-leaf boundary: PASS
RFC seven-leaf example: PASS

=== INCLUSION VERIFICATION ===
Positive tests passed: 2145
Negative tests passed: 8579
Total inclusion tests: 10724

=== TREE BOUNDARIES ===
Empty root: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
First root: 6bb5021c69bfac9762378ec9697134da0542f86caf798f4fc1fbeee068ddeac4
Final root: b4aacc347149986abf4f54694aa5877c740931b34d57701ca31e9c701b7a14e3

VALIDATION: PASS


In [6]:
# ============================================================
# CELL 5 — Append-only Merkle consistency proofs
# ============================================================
# Implement consistency proof generation and verification.
#
# Reference: RFC 9162, Sections 2.1.4.1, 2.1.4.2 and 2.1.5.
# https://doi.org/10.17487/RFC9162
#
# Empty and equal-size trees are handled explicitly as
# benchmark boundary conditions.


def consistency_subproof(entries, old_size, known_root):
    """Recursively construct a Merkle consistency path."""

    if old_size == len(entries):
        return [] if known_root else [log_tree_root(entries)]

    k = log_split(len(entries))

    if old_size <= k:
        return consistency_subproof(
            entries[:k], old_size, known_root
        ) + [log_tree_root(entries[k:])]

    return consistency_subproof(
        entries[k:], old_size - k, False
    ) + [log_tree_root(entries[:k])]


def log_consistency_proof(entries, old_size):
    """Generate a proof of append-only extension."""

    if not 0 <= old_size <= len(entries):
        raise ValueError("Invalid historical tree size.")

    if old_size == 0:
        return []

    return consistency_subproof(entries, old_size, True)


def verify_log_consistency(
    old_size, new_size, old_root, new_root, proof
):
    """Verify consistency between two Merkle checkpoints."""

    if not 0 <= old_size <= new_size:
        return False

    if any(
        type(node) is not bytes or len(node) != 32
        for node in [old_root, new_root, *proof]
    ):
        return False

    # Empty tree: consistency is trivial, but the newer
    # checkpoint still requires independent authentication.
    if old_size == 0:
        return (
            old_root == log_tree_root([])
            and not proof
            and (new_size != 0 or new_root == old_root)
        )

    # Equal-sized trees must have identical roots.
    if old_size == new_size:
        return old_root == new_root and not proof

    if not proof:
        return False

    # RFC 9162: prepend the known historical root
    # when its tree size is an exact power of two.
    if old_size & (old_size - 1) == 0:
        path = [old_root, *proof]
    else:
        path = list(proof)

    first = old_size - 1
    second = new_size - 1

    while first & 1:
        first >>= 1
        second >>= 1

    reconstructed_old = path[0]
    reconstructed_new = path[0]

    for node in path[1:]:

        if second == 0:
            return False

        if first & 1 or first == second:

            reconstructed_old = log_node_hash(
                node, reconstructed_old
            )

            reconstructed_new = log_node_hash(
                node, reconstructed_new
            )

            while first and not (first & 1):
                first >>= 1
                second >>= 1

        else:

            reconstructed_new = log_node_hash(
                reconstructed_new, node
            )

        first >>= 1
        second >>= 1

    return (
        second == 0
        and reconstructed_old == old_root
        and reconstructed_new == new_root
    )


def flip_digest_bit(digest):
    """Introduce a controlled one-bit modification."""

    return bytes([digest[0] ^ 1]) + digest[1:]


# ------------------------------------------------------------
# RFC 9162 reference examples
# ------------------------------------------------------------

example = [f"d{i}".encode() for i in range(7)]

assert log_consistency_proof(example, 3) == [
    log_leaf_hash(example[2]),
    log_leaf_hash(example[3]),
    log_tree_root(example[:2]),
    log_tree_root(example[4:]),
]

assert log_consistency_proof(example, 4) == [
    log_tree_root(example[4:])
]

assert log_consistency_proof(example, 6) == [
    log_tree_root(example[4:6]),
    log_leaf_hash(example[6]),
    log_tree_root(example[:4]),
]


# ------------------------------------------------------------
# Boundary condition tests
# ------------------------------------------------------------

empty_root = log_roots[0]
first_root = log_roots[1]

assert verify_log_consistency(
    0, 0, empty_root, empty_root, []
)

assert verify_log_consistency(
    0, 1, empty_root, first_root, []
)

assert verify_log_consistency(
    1, 1, first_root, first_root, []
)

assert not verify_log_consistency(
    1, 1, first_root, flip_digest_bit(first_root), []
)

assert not verify_log_consistency(
    1, 0, first_root, empty_root, []
)


# ------------------------------------------------------------
# Exhaustive development consistency tests
# ------------------------------------------------------------

positive_consistency_tests = 0
negative_consistency_tests = 0

for new_size in range(1, len(log_entries) + 1):

    entries = log_entries[:new_size]
    new_root = log_roots[new_size]

    for old_size in range(1, new_size):

        old_root = log_roots[old_size]

        proof = log_consistency_proof(entries, old_size)

        # Positive: valid append-only extension.
        assert verify_log_consistency(
            old_size, new_size, old_root, new_root, proof
        )

        positive_consistency_tests += 1

        # Negative 1: modified historical root.
        assert not verify_log_consistency(
            old_size, new_size,
            flip_digest_bit(old_root), new_root, proof
        )

        # Negative 2: modified newer root.
        assert not verify_log_consistency(
            old_size, new_size,
            old_root, flip_digest_bit(new_root), proof
        )

        # Negative 3: incomplete proof.
        assert not verify_log_consistency(
            old_size, new_size,
            old_root, new_root, proof[:-1]
        )

        # Negative 4: excessive proof length.
        assert not verify_log_consistency(
            old_size, new_size,
            old_root, new_root, proof + [bytes(32)]
        )

        # Negative 5: modified proof node.
        modified_proof = [
            flip_digest_bit(proof[0]), *proof[1:]
        ]

        assert not verify_log_consistency(
            old_size, new_size,
            old_root, new_root, modified_proof
        )

        # Negative 6: internally valid but conflicting history.
        conflicting_entries = [
            flip_digest_bit(entries[0]), *entries[1:]
        ]

        conflicting_root = log_tree_root(conflicting_entries)

        conflicting_proof = log_consistency_proof(
            conflicting_entries, old_size
        )

        assert not verify_log_consistency(
            old_size, new_size,
            old_root, conflicting_root, conflicting_proof
        )

        negative_consistency_tests += 6


# ------------------------------------------------------------
# Report validation results
# ------------------------------------------------------------

print("=== CONSISTENCY PROOF VALIDATION ===")
print("RFC reference examples: 3/3 PASS")
print("Boundary condition tests: 5/5 PASS")

print("\n=== DEVELOPMENT CONSISTENCY TESTS ===")
print("Log entries:", len(log_entries))
print("Positive tests:", positive_consistency_tests)
print("Negative tests:", negative_consistency_tests)

print("Total consistency tests:",
      positive_consistency_tests + negative_consistency_tests)

print("\n=== SECURITY PROPERTIES ===")
print("Valid append-only extensions: PASS")
print("Modified checkpoint rejection: PASS")
print("Incomplete proof rejection: PASS")
print("Excessive proof rejection: PASS")
print("Modified proof rejection: PASS")
print("Conflicting history rejection: PASS")

print("\nVALIDATION: PASS")

=== CONSISTENCY PROOF VALIDATION ===
RFC reference examples: 3/3 PASS
Boundary condition tests: 5/5 PASS

=== DEVELOPMENT CONSISTENCY TESTS ===
Log entries: 65
Positive tests: 2080
Negative tests: 12480
Total consistency tests: 14560

=== SECURITY PROPERTIES ===
Valid append-only extensions: PASS
Modified checkpoint rejection: PASS
Incomplete proof rejection: PASS
Excessive proof rejection: PASS
Modified proof rejection: PASS
Conflicting history rejection: PASS

VALIDATION: PASS


In [7]:
# ============================================================
# CELL 6 — Stateful witnesses and quorum verification
# ============================================================
# Implement authenticated checkpoints, stateful witnesses
# and the approved 3-of-4 witness policy.
#
# References:
# C2SP Transparency Log Witness Protocol v1.1.0
# https://c2sp.org/tlog-witness@v1.1.0
#
# RFC 9162, Section 2.1.4
# https://doi.org/10.17487/RFC9162
#
# This is a controlled research implementation.
# It is not a production witness service.

from itertools import combinations
import re

# Approved configuration: EXPERIMENTAL_DESIGN.md, Section 4.
WITNESS_COUNT = 4
WITNESS_QUORUM = 3
MAX_COMPROMISED_WITNESSES = 1

LOG_ID = "iot-evidence-transparency-log-v1"

OPERATOR_DOMAIN = b"iot-benchmark:log-checkpoint:v1\x00"
WITNESS_DOMAIN = b"iot-benchmark:witness-checkpoint:v1\x00"

operator_key = experimental_key("log-operator")
operator_public_key = public_key_bytes(operator_key)

witness_keys = {
    f"W{i}": experimental_key(f"witness:{i}")
    for i in range(1, WITNESS_COUNT + 1)
}

witness_public_keys = {
    name: public_key_bytes(key)
    for name, key in witness_keys.items()
}

assert len(set(witness_public_keys.values())) == WITNESS_COUNT


def make_checkpoint(entries):
    """Construct a checkpoint for the configured log."""

    return {
        "log_id": LOG_ID,
        "size": len(entries),
        "root": log_tree_root(entries).hex(),
    }


def valid_checkpoint(checkpoint):
    """Validate the checkpoint representation and log identity."""

    return (
        type(checkpoint) is dict
        and set(checkpoint) == {"log_id", "size", "root"}
        and checkpoint["log_id"] == LOG_ID
        and type(checkpoint["size"]) is int
        and checkpoint["size"] >= 0
        and type(checkpoint["root"]) is str
        and re.fullmatch(r"[0-9a-f]{64}", checkpoint["root"])
        is not None
        and (
            checkpoint["size"] != 0
            or checkpoint["root"] == log_tree_root([]).hex()
        )
    )


def sign_checkpoint(checkpoint, key, domain):
    """Sign a checkpoint using a role-specific signature domain."""

    message = domain + canonical_bytes(checkpoint)

    return eddsa.new(key, "rfc8032").sign(message)


def verify_checkpoint_signature(
    checkpoint, signature, public_key, domain
):
    """Verify checkpoint structure and signature."""

    if not valid_checkpoint(checkpoint):
        return False

    try:
        key = eddsa.import_public_key(public_key)

        eddsa.new(key, "rfc8032").verify(
            domain + canonical_bytes(checkpoint),
            signature,
        )

        return True

    except (ValueError, TypeError):
        return False


class Witness:
    """A stateful witness with independently retained history."""

    def __init__(self, name):

        self.name = name
        self.checkpoint = make_checkpoint([])

    def cosign(self, checkpoint, operator_signature, proof):

        # Authenticate the log operator.
        if not verify_checkpoint_signature(
            checkpoint,
            operator_signature,
            operator_public_key,
            OPERATOR_DOMAIN,
        ):
            return None

        # Verify consistency against retained witness state.
        if not verify_log_consistency(
            self.checkpoint["size"],
            checkpoint["size"],
            bytes.fromhex(self.checkpoint["root"]),
            bytes.fromhex(checkpoint["root"]),
            proof,
        ):
            return None

        # Sign and retain the newly verified checkpoint.
        signature = sign_checkpoint(
            checkpoint,
            witness_keys[self.name],
            WITNESS_DOMAIN,
        )

        self.checkpoint = deepcopy(checkpoint)

        return self.name, signature


def verify_quorum(checkpoint, operator_signature, cosignatures):
    """Require signatures from three distinct authorized witnesses."""

    if not verify_checkpoint_signature(
        checkpoint,
        operator_signature,
        operator_public_key,
        OPERATOR_DOMAIN,
    ):
        return False

    valid_witnesses = set()

    for name, signature in cosignatures:

        if name not in witness_public_keys:
            continue

        if verify_checkpoint_signature(
            checkpoint,
            signature,
            witness_public_keys[name],
            WITNESS_DOMAIN,
        ):
            valid_witnesses.add(name)

    return len(valid_witnesses) >= WITNESS_QUORUM


# ------------------------------------------------------------
# Validate the approved quorum configuration
# ------------------------------------------------------------

possible_quorums = list(
    combinations(witness_keys, WITNESS_QUORUM)
)

minimum_intersection = min(
    len(set(first) & set(second))
    for first, second in combinations(possible_quorums, 2)
)

assert minimum_intersection > MAX_COMPROMISED_WITNESSES


# ------------------------------------------------------------
# Construct witnessed development checkpoints
# ------------------------------------------------------------

witnesses = {
    name: Witness(name)
    for name in witness_keys
}

witnessed_checkpoints = []

for size in range(1, len(log_entries) + 1):

    entries = log_entries[:size]
    checkpoint = make_checkpoint(entries)

    operator_signature = sign_checkpoint(
        checkpoint, operator_key, OPERATOR_DOMAIN
    )

    cosignatures = []

    for witness in witnesses.values():

        proof = log_consistency_proof(
            entries, witness.checkpoint["size"]
        )

        result = witness.cosign(
            checkpoint, operator_signature, proof
        )

        assert result is not None

        cosignatures.append(result)

    assert verify_quorum(
        checkpoint, operator_signature, cosignatures
    )

    witnessed_checkpoints.append({
        "checkpoint": checkpoint,
        "operator_signature": operator_signature,
        "cosignatures": cosignatures,
    })


# ------------------------------------------------------------
# Quorum and availability tests
# ------------------------------------------------------------

sample = witnessed_checkpoints[-1]

checkpoint = sample["checkpoint"]
operator_signature = sample["operator_signature"]
cosignatures = sample["cosignatures"]

# Test all possible three-witness configurations.
for subset in combinations(cosignatures, WITNESS_QUORUM):

    assert verify_quorum(
        checkpoint, operator_signature, subset
    )

# Two witnesses must be insufficient.
for subset in combinations(cosignatures, WITNESS_QUORUM - 1):

    assert not verify_quorum(
        checkpoint, operator_signature, subset
    )

# Repeating one witness signature cannot create a quorum.
assert not verify_quorum(
    checkpoint,
    operator_signature,
    [cosignatures[0]] * WITNESS_QUORUM,
)

# A valid signature under another role is not a witness vote.
assert not verify_quorum(
    checkpoint,
    operator_signature,
    [
        *cosignatures[:2],
        (cosignatures[2][0], operator_signature),
    ],
)

# A modified operator signature must be rejected.
assert not verify_quorum(
    checkpoint,
    flip_digest_bit(operator_signature),
    cosignatures,
)

# Witnesses reject an older checkpoint.
older = witnessed_checkpoints[-2]

assert witnesses["W1"].cosign(
    older["checkpoint"],
    older["operator_signature"],
    [],
) is None


# ------------------------------------------------------------
# Controlled conflicting-history test
# ------------------------------------------------------------

# Construct two individually valid but conflicting histories.
history_a = log_entries[:2]

history_b = [
    flip_digest_bit(history_a[0]),
    history_a[1],
]

checkpoint_a = make_checkpoint(history_a)
checkpoint_b = make_checkpoint(history_b)

assert checkpoint_a["root"] != checkpoint_b["root"]

signature_a = sign_checkpoint(
    checkpoint_a, operator_key, OPERATOR_DOMAIN
)

signature_b = sign_checkpoint(
    checkpoint_b, operator_key, OPERATOR_DOMAIN
)

# Reset witness state for an independent scenario.
fork_witnesses = {
    name: Witness(name)
    for name in witness_keys
}

# W1 is modelled as compromised and signs both histories.
compromised_vote_a = (
    "W1",
    sign_checkpoint(
        checkpoint_a, witness_keys["W1"], WITNESS_DOMAIN
    ),
)

compromised_vote_b = (
    "W1",
    sign_checkpoint(
        checkpoint_b, witness_keys["W1"], WITNESS_DOMAIN
    ),
)

# Two honest witnesses accept history A.
vote_a2 = fork_witnesses["W2"].cosign(
    checkpoint_a, signature_a, []
)

vote_a3 = fork_witnesses["W3"].cosign(
    checkpoint_a, signature_a, []
)

assert vote_a2 is not None
assert vote_a3 is not None

assert verify_quorum(
    checkpoint_a,
    signature_a,
    [compromised_vote_a, vote_a2, vote_a3],
)

# Previously committed honest witnesses reject history B.
for name in ("W2", "W3"):

    assert fork_witnesses[name].cosign(
        checkpoint_b, signature_b, []
    ) is None

# The fourth witness has not previously observed history A.
vote_b4 = fork_witnesses["W4"].cosign(
    checkpoint_b, signature_b, []
)

assert vote_b4 is not None

# The conflicting history obtains only two votes.
assert not verify_quorum(
    checkpoint_b,
    signature_b,
    [compromised_vote_b, vote_b4],
)


# ------------------------------------------------------------
# Report validation results
# ------------------------------------------------------------

print("=== WITNESS CONFIGURATION ===")
print("Registered witnesses:", len(witnesses))
print("Required quorum:", WITNESS_QUORUM)
print("Maximum assumed compromised witnesses:",
      MAX_COMPROMISED_WITNESSES)
print("Possible quorums:", len(possible_quorums))
print("Minimum quorum intersection:", minimum_intersection)

print("\n=== DEVELOPMENT CHECKPOINTS ===")
print("Witnessed checkpoints:", len(witnessed_checkpoints))
print("Valid witness signatures:", sum(
    len(item["cosignatures"])
    for item in witnessed_checkpoints
))
print("Accepted checkpoint quorums:",
      len(witnessed_checkpoints))

print("\n=== CONTROLLED SECURITY TESTS ===")
print("Three-witness availability: PASS")
print("Two-witness rejection: PASS")
print("Duplicate witness rejection: PASS")
print("Signature domain separation: PASS")
print("Modified operator signature rejection: PASS")
print("Historical checkpoint rollback rejection: PASS")
print("Conflicting checkpoint rejection: PASS")
print("Single compromised witness fork test: PASS")

print("\nVALIDATION: PASS")

=== WITNESS CONFIGURATION ===
Registered witnesses: 4
Required quorum: 3
Maximum assumed compromised witnesses: 1
Possible quorums: 4
Minimum quorum intersection: 2

=== DEVELOPMENT CHECKPOINTS ===
Witnessed checkpoints: 65
Valid witness signatures: 260
Accepted checkpoint quorums: 65

=== CONTROLLED SECURITY TESTS ===
Three-witness availability: PASS
Two-witness rejection: PASS
Duplicate witness rejection: PASS
Signature domain separation: PASS
Modified operator signature rejection: PASS
Historical checkpoint rollback rejection: PASS
Conflicting checkpoint rejection: PASS
Single compromised witness fork test: PASS

VALIDATION: PASS


In [8]:
# ============================================================
# CELL 7 — Local Ethereum anchoring architecture
# ============================================================
# Compile, deploy and test an append-only commitment contract.
#
# References:
# https://pypi.org/project/eth-tester/0.14.0b1/
# https://docs.soliditylang.org/en/v0.8.36/
#
# Local EVM execution does not establish decentralized
# consensus security or public-network performance.

import subprocess
from statistics import median
from eth_tester import EthereumTester, PyEVMBackend
from eth_tester.exceptions import TransactionFailed
from eth_hash.auto import keccak
from eth_abi import encode, decode

COMPILER_DIR = ROOT / "tools" / "solidity"
SOLIDITY_VERSION = "0.8.36"

# Execution budget, not a measured transaction cost.
GAS_LIMIT = 300_000
ZERO_COMMITMENT = bytes(32)

CONTRACT_SOURCE = """
// SPDX-License-Identifier: MIT
pragma solidity 0.8.36;

contract EvidenceAnchor {

    address public immutable relay;

    uint256 public count;
    bytes32 public head;

    mapping(uint256 => bytes32) public commitments;

    event Anchored(
        uint256 indexed sequence,
        bytes32 commitment,
        bytes32 previousCommitment
    );

    constructor() {
        relay = msg.sender;
    }

    function append(
        uint256 sequence,
        bytes32 commitment,
        bytes32 previousCommitment
    ) external {

        require(msg.sender == relay, "unauthorized relay");
        require(sequence == count, "invalid sequence");
        require(previousCommitment == head, "invalid predecessor");
        require(commitment != bytes32(0), "empty commitment");

        commitments[sequence] = commitment;

        head = commitment;
        count += 1;

        emit Anchored(
            sequence,
            commitment,
            previousCommitment
        );
    }
}
"""


# ------------------------------------------------------------
# Compile using the preserved Solidity environment
# ------------------------------------------------------------

def compile_anchor_contract():

    compiler_version = subprocess.check_output(
        ["node", "-e", "console.log(require('solc').version())"],
        cwd=COMPILER_DIR,
        text=True,
    ).strip()

    assert compiler_version.startswith(SOLIDITY_VERSION + "+")

    compiler_input = {
        "language": "Solidity",
        "sources": {
            "EvidenceAnchor.sol": {
                "content": CONTRACT_SOURCE
            }
        },
        "settings": {
            "evmVersion": "cancun",
            "optimizer": {"enabled": False},
            "outputSelection": {
                "*": {
                    "*": ["abi", "evm.bytecode.object"]
                }
            },
        },
    }

    output = subprocess.check_output(
        [
            "node", "-e",
            """
            const fs = require('fs');
            const solc = require('solc');
            process.stdout.write(
                solc.compile(fs.readFileSync(0, 'utf8'))
            );
            """,
        ],
        cwd=COMPILER_DIR,
        input=json.dumps(compiler_input),
        text=True,
    )

    compiled = json.loads(output)

    errors = [
        item for item in compiled.get("errors", [])
        if item["severity"] == "error"
    ]

    assert not errors, errors

    contract = compiled["contracts"][
        "EvidenceAnchor.sol"
    ]["EvidenceAnchor"]

    return contract["evm"]["bytecode"]["object"]


anchor_bytecode = compile_anchor_contract()


# ------------------------------------------------------------
# Local Ethereum interface
# ------------------------------------------------------------

def contract_calldata(function, types=(), values=()):
    """Encode a Solidity function call."""

    selector = keccak(function.encode("utf-8"))[:4]

    arguments = encode(list(types), list(values))

    return "0x" + (selector + arguments).hex()


class EthereumAnchor:
    """Execute and query the commitment contract locally."""

    def __init__(self):

        self.tester = EthereumTester(
            backend=PyEVMBackend()
        )

        self.relay = self.tester.get_accounts()[0]
        self.other_account = self.tester.get_accounts()[1]

        deployment = {
            "from": self.relay,
            "data": "0x" + anchor_bytecode,
        }

        deployment["gas"] = self.tester.estimate_gas(
            deployment
        )

        tx_hash = self.tester.send_transaction(deployment)

        self.deployment_receipt = (
            self.tester.get_transaction_receipt(tx_hash)
        )

        assert self.deployment_receipt["status"] == 1

        self.address = (
            self.deployment_receipt["contract_address"]
        )

    def query(self, function, result_type, types=(), values=()):

        result = self.tester.call({
            "from": self.relay,
            "to": self.address,
            "data": contract_calldata(
                function, types, values
            ),
        })

        return decode(
            [result_type],
            bytes.fromhex(result[2:])
        )[0]

    def count(self):
        return self.query("count()", "uint256")

    def head(self):
        return self.query("head()", "bytes32")

    def commitment_at(self, sequence):

        if not 0 <= sequence < self.count():
            return None

        return self.query(
            "commitments(uint256)",
            "bytes32",
            ["uint256"],
            [sequence],
        )

    def append_transaction(
        self, sequence, commitment, previous, sender=None
    ):

        return {
            "from": sender or self.relay,
            "to": self.address,
            "data": contract_calldata(
                "append(uint256,bytes32,bytes32)",
                ["uint256", "bytes32", "bytes32"],
                [sequence, commitment, previous],
            ),
            "gas": GAS_LIMIT,
        }

    def append(
        self, sequence, commitment, previous, sender=None
    ):

        transaction = self.append_transaction(
            sequence, commitment, previous, sender
        )

        tx_hash = self.tester.send_transaction(transaction)

        return self.tester.get_transaction_receipt(tx_hash)


# ------------------------------------------------------------
# Deploy and validate initial state
# ------------------------------------------------------------

ethereum_anchor = EthereumAnchor()

assert ethereum_anchor.count() == 0
assert ethereum_anchor.head() == ZERO_COMMITMENT
assert ethereum_anchor.commitment_at(0) is None

assert ethereum_anchor.query(
    "relay()", "address"
).lower() == ethereum_anchor.relay.lower()


# ------------------------------------------------------------
# Anchor all development evidence
# ------------------------------------------------------------

ethereum_receipts = []

event_topic = "0x" + keccak(
    b"Anchored(uint256,bytes32,bytes32)"
).hex()

for item in signed_development_evidence:

    evidence = item["evidence"]

    # Apply the same source authentication used by A and B.
    assert verify_signature(
        evidence, item["signature"], item["public_key"]
    )

    assert is_authorized(evidence, item["public_key"])

    sequence = evidence["sequence"]

    commitment = bytes.fromhex(
        evidence_commitment(evidence)
    )

    assert commitment.hex() == item["commitment"]

    previous = (
        ZERO_COMMITMENT
        if evidence["previous_commitment"] is None
        else bytes.fromhex(evidence["previous_commitment"])
    )

    receipt = ethereum_anchor.append(
        sequence, commitment, previous
    )

    assert receipt["status"] == 1

    # Verify actual contract storage.
    assert ethereum_anchor.commitment_at(sequence) == commitment
    assert ethereum_anchor.head() == commitment
    assert ethereum_anchor.count() == sequence + 1

    # Independently inspect the emitted event.
    assert len(receipt["logs"]) == 1

    event = receipt["logs"][0]

    assert event["address"].lower() == (
        ethereum_anchor.address.lower()
    )

    assert len(event["topics"]) == 2
    assert event["topics"][0] == event_topic
    assert int(event["topics"][1], 16) == sequence

    event_commitment, event_previous = decode(
        ["bytes32", "bytes32"],
        bytes.fromhex(event["data"][2:])
    )

    assert event_commitment == commitment
    assert event_previous == previous

    ethereum_receipts.append(receipt)


# ------------------------------------------------------------
# Verify failed transactions and unchanged state
# ------------------------------------------------------------

def anchor_state(anchor):
    """Read the complete current commitment history."""

    return (
        anchor.count(),
        anchor.head(),
        tuple(
            anchor.commitment_at(i)
            for i in range(anchor.count())
        ),
    )


original_state = anchor_state(ethereum_anchor)

next_sequence = ethereum_anchor.count()
current_head = ethereum_anchor.head()

candidate_commitment = hashlib.sha256(
    b"development-negative-test"
).digest()

# Confirm that the gas budget is sufficient for a valid append.
valid_transaction = ethereum_anchor.append_transaction(
    next_sequence, candidate_commitment, current_head
)

assert ethereum_anchor.tester.estimate_gas(
    valid_transaction
) < GAS_LIMIT


def check_rejected_append(
    name, sequence, commitment, previous,
    expected_reason, sender=None
):

    transaction = ethereum_anchor.append_transaction(
        sequence, commitment, previous, sender
    )

    # Verify the intended failure reason before execution.
    try:
        ethereum_anchor.tester.call(transaction)

    except TransactionFailed as error:
        assert expected_reason in str(error), str(error)

    else:
        raise AssertionError("Invalid append did not revert.")

    # Submit an actual transaction and inspect its receipt.
    tx_hash = ethereum_anchor.tester.send_transaction(
        transaction
    )

    receipt = ethereum_anchor.tester.get_transaction_receipt(
        tx_hash
    )

    assert receipt["status"] == 0
    assert receipt["gas_used"] < GAS_LIMIT
    assert not receipt["logs"]

    # A failed append must not modify any stored commitment.
    assert anchor_state(ethereum_anchor) == original_state

    print(f"{name}: PASS")


print("=== CONTROLLED CONTRACT TESTS ===")

check_rejected_append(
    "Duplicate sequence rejection",
    0, log_entries[0], ZERO_COMMITMENT,
    "invalid sequence",
)

check_rejected_append(
    "Existing commitment overwrite rejection",
    0, candidate_commitment, ZERO_COMMITMENT,
    "invalid sequence",
)

check_rejected_append(
    "Skipped sequence rejection",
    next_sequence + 1, candidate_commitment, current_head,
    "invalid sequence",
)

check_rejected_append(
    "Incorrect predecessor rejection",
    next_sequence, candidate_commitment,
    flip_digest_bit(current_head),
    "invalid predecessor",
)

check_rejected_append(
    "Empty commitment rejection",
    next_sequence, ZERO_COMMITMENT, current_head,
    "empty commitment",
)

check_rejected_append(
    "Unauthorized relay rejection",
    next_sequence, candidate_commitment, current_head,
    "unauthorized relay",
    sender=ethereum_anchor.other_account,
)


# ------------------------------------------------------------
# Final comparison against the common commitment history
# ------------------------------------------------------------

assert ethereum_anchor.count() == len(log_entries)

assert [
    ethereum_anchor.commitment_at(i)
    for i in range(len(log_entries))
] == log_entries

assert ethereum_anchor.head() == log_entries[-1]

gas_values = [
    receipt["gas_used"] for receipt in ethereum_receipts
]

print("\n=== ETHEREUM ANCHORING ===")
print("Contract deployment: PASS")
print("Anchored commitments:", ethereum_anchor.count())
print("Successful transactions:", len(ethereum_receipts))
print("Matching stored commitments:", len(log_entries))
print("Verified anchor events:", len(ethereum_receipts))
print("Final commitment:", ethereum_anchor.head().hex())

print("\n=== EXECUTION GAS ===")
print("Deployment:", ethereum_anchor.deployment_receipt["gas_used"])
print("Minimum append:", min(gas_values))
print("Median append:", median(gas_values))
print("Maximum append:", max(gas_values))
print("Total append gas:", sum(gas_values))

print("\n=== FINAL VALIDATION ===")
print("Controlled rejection tests: 6/6 PASS")
print("Historical state preservation: PASS")
print("Common commitment equivalence: PASS")

print("\nVALIDATION: PASS")

=== CONTROLLED CONTRACT TESTS ===
Duplicate sequence rejection: PASS
Existing commitment overwrite rejection: PASS
Skipped sequence rejection: PASS
Incorrect predecessor rejection: PASS
Empty commitment rejection: PASS
Unauthorized relay rejection: PASS

=== ETHEREUM ANCHORING ===
Contract deployment: PASS
Anchored commitments: 65
Successful transactions: 65
Matching stored commitments: 65
Verified anchor events: 65
Final commitment: b95e58c148f97265dfef56a9fd718d3e38e357a458d629112d443a6f1638f2ec

=== EXECUTION GAS ===
Deployment: 440947
Minimum append: 57530
Median append: 57554
Maximum append: 91346
Total append gas: 3774670

=== FINAL VALIDATION ===
Controlled rejection tests: 6/6 PASS
Historical state preservation: PASS
Common commitment equivalence: PASS

VALIDATION: PASS


In [9]:
# ============================================================
# CELL 8 — Independent evidence-policy context
# ============================================================
# Construct independent synthetic reference conditions.
#
# Source: PROTOCOL_AMENDMENT_02.md, Sections 2–5.
#
# Policy context is derived from original development data,
# not from submitted or subsequently modified evidence.

from dataclasses import dataclass
from datetime import timedelta

POLICY_VERSION = "iot-evidence-policy-v1"

# Independent policy specification from the approved amendment.
# Do not derive required units from submitted evidence.
POLICY_UNITS = (
    ("voltage", "V"),
    ("current", "A"),
    ("power", "W"),
    ("energy_Wh", "Wh"),
    ("cumulative_energy_Wh", "Wh"),
)

POLICY_SCOPE = "submit_measurement_evidence"


@dataclass(frozen=True)
class EvidencePolicyContext:
    """Immutable synthetic reference conditions."""

    expected_sequence: int
    expected_source_id: str
    expected_record_count: int
    window_start: datetime
    window_end: datetime
    expected_units: tuple
    required_scope: str


def original_measurement_time(value):
    """Interpret original logger measurement times as UTC."""

    timestamp = datetime.fromisoformat(value)

    # Original measurement timestamps are timezone-naive.
    assert timestamp.tzinfo is None

    return timestamp.replace(tzinfo=timezone.utc)


def build_policy_context(batch):
    """Construct policy context independently of submitted evidence."""

    records = batch["records"]
    proof = batch["proof"]

    assert records

    source_ids = {
        record["device_id"] for record in records
    }

    assert len(source_ids) == 1

    window_start = original_measurement_time(
        records[0]["timestamp"]
    )

    window_end = datetime.fromisoformat(
        proof["timestamp"]
    )

    assert window_end.utcoffset() == timedelta(0)

    assert window_start <= window_end

    assert proof["size"] == len(records)

    return EvidencePolicyContext(
        expected_sequence=batch["batch_index"],
        expected_source_id=next(iter(source_ids)),
        expected_record_count=proof["size"],
        window_start=window_start,
        window_end=window_end,
        expected_units=POLICY_UNITS,
        required_scope=POLICY_SCOPE,
    )


# ------------------------------------------------------------
# Construct development policy contexts
# ------------------------------------------------------------

development_policy_contexts = {
    batch["batch_index"]: build_policy_context(batch)
    for batch in development_batches
}

assert len(development_policy_contexts) == DEVELOPMENT_COUNT


# ------------------------------------------------------------
# Validate compatibility with unmodified development controls
# ------------------------------------------------------------

context_tests = 0

for item in signed_development_evidence:

    evidence = item["evidence"]

    context = development_policy_contexts[
        evidence["sequence"]
    ]

    records = evidence["records"]

    first_time = original_measurement_time(
        records[0]["timestamp"]
    )

    last_time = original_measurement_time(
        records[-1]["timestamp"]
    )

    application_time = datetime.fromisoformat(
        evidence["claims"]["application_timestamp_utc"]
    )

    assert evidence["sequence"] == context.expected_sequence

    assert evidence["source_id"] == context.expected_source_id

    assert len(records) == context.expected_record_count

    assert evidence["claims"]["measurement_units"] == (
        dict(context.expected_units)
    )

    assert (
        context.window_start
        <= first_time
        <= last_time
        <= application_time
        <= context.window_end
    )

    context_tests += 1


# ------------------------------------------------------------
# Verify separation between evidence and policy context
# ------------------------------------------------------------

sample = deepcopy(
    signed_development_evidence[0]["evidence"]
)

# Context is selected by the intended experimental case,
# not by an untrusted submitted sequence identifier.
target_sequence = development_batches[0]["batch_index"]

context = development_policy_contexts[target_sequence]

# Introduce modifications to a copied evidence object.
sample["records"].pop()
sample["sequence"] += 1
sample["source_id"] = "unauthorized_device"
sample["claims"]["measurement_units"]["current"] = "mA"

sample["claims"]["application_timestamp_utc"] = (
    context.window_start - timedelta(microseconds=1)
).isoformat()

# Modifying evidence must not modify policy conditions.
assert len(sample["records"]) != context.expected_record_count

assert sample["sequence"] != context.expected_sequence

assert sample["source_id"] != context.expected_source_id

assert sample["claims"]["measurement_units"] != (
    dict(context.expected_units)
)

assert datetime.fromisoformat(
    sample["claims"]["application_timestamp_utc"]
) < context.window_start

# Reconstruct the original context independently.
assert context == build_policy_context(
    development_batches[0]
)


# ------------------------------------------------------------
# Report validation results
# ------------------------------------------------------------

window_durations = [
    (context.window_end - context.window_start).total_seconds()
    for context in development_policy_contexts.values()
]

print("=== EVIDENCE POLICY CONFIGURATION ===")
print("Policy version:", POLICY_VERSION)
print("Development contexts:", len(development_policy_contexts))
print("Required measurement units:", len(POLICY_UNITS))
print("Required certificate scope:", POLICY_SCOPE)

print("\n=== REFERENCE CONDITIONS ===")
print("Expected total measurements:", sum(
    context.expected_record_count
    for context in development_policy_contexts.values()
))

print("Minimum window duration (s):", min(window_durations))
print("Median window duration (s):", median(window_durations))
print("Maximum window duration (s):", max(window_durations))

print("\n=== DEVELOPMENT CONTROL VALIDATION ===")
print("Compatible development controls:", context_tests)
print("Expected sequence consistency: PASS")
print("Expected record count consistency: PASS")
print("Expected source consistency: PASS")
print("Measurement unit consistency: PASS")
print("Temporal window consistency: PASS")

print("\n=== POLICY INDEPENDENCE ===")
print("Record omission independence: PASS")
print("Sequence modification independence: PASS")
print("Source modification independence: PASS")
print("Unit modification independence: PASS")
print("Timestamp modification independence: PASS")
print("Deterministic context reconstruction: PASS")

print("\nVALIDATION: PASS")

=== EVIDENCE POLICY CONFIGURATION ===
Policy version: iot-evidence-policy-v1
Development contexts: 65
Required measurement units: 5
Required certificate scope: submit_measurement_evidence

=== REFERENCE CONDITIONS ===
Expected total measurements: 3874
Minimum window duration (s): 1381.586674
Median window duration (s): 1921.538782
Maximum window duration (s): 35041.798044

=== DEVELOPMENT CONTROL VALIDATION ===
Compatible development controls: 65
Expected sequence consistency: PASS
Expected record count consistency: PASS
Expected source consistency: PASS
Measurement unit consistency: PASS
Temporal window consistency: PASS

=== POLICY INDEPENDENCE ===
Record omission independence: PASS
Sequence modification independence: PASS
Source modification independence: PASS
Unit modification independence: PASS
Timestamp modification independence: PASS
Deterministic context reconstruction: PASS

VALIDATION: PASS


In [11]:
# ============================================================
# CELL 9 — Institutional certificate mechanism
# ============================================================
# Implement R09 under the approved synthetic evidence policy.
#
# References:
# PROTOCOL_AMENDMENT_02.md, Section 5.
# W3C Verifiable Credentials Data Model v2.0,
# Sections 4.7–4.9 and 5.1.
# https://www.w3.org/TR/2025/REC-vc-data-model-2.0-20250515/
#
# Certificates and keys are synthetic experimental fixtures.
# This is not a production certification system.

CERTIFICATE_VERSION = "iot-authorization-certificate-v1"

CERTIFICATE_DOMAIN = (
    CERTIFICATE_VERSION.encode("utf-8") + b"\x00"
)

ISSUER_ID = "experimental_certification_authority"

issuer_private_key = experimental_key(
    "institutional-certificate-issuer"
)

untrusted_issuer_key = experimental_key(
    "untrusted-certificate-issuer"
)

# Independent trusted issuer configuration.
trusted_issuers = {
    ISSUER_ID: public_key_bytes(issuer_private_key)
}

assert trusted_issuers[ISSUER_ID] not in (
    authorized_sources.values()
)

CERTIFICATE_FIELDS = {
    "schema",
    "issuer",
    "subject",
    "subject_public_key",
    "valid_from",
    "valid_until",
    "scope",
    "metadata_origin",
}

CERTIFICATE_CHECKS = (
    "structure",
    "issuer",
    "signature",
    "subject",
    "key_binding",
    "scope",
    "validity",
)


def sign_certificate(body, private_key):
    """Create an independently signed certificate."""

    signature = eddsa.new(
        private_key, "rfc8032"
    ).sign(
        CERTIFICATE_DOMAIN + canonical_bytes(body)
    )

    return {
        "body": deepcopy(body),
        "signature": signature,
    }


def build_certificate(context):
    """Issue a synthetic certificate from independent context."""

    body = {
        "schema": CERTIFICATE_VERSION,
        "issuer": ISSUER_ID,
        "subject": context.expected_source_id,
        "subject_public_key": authorized_sources[
            context.expected_source_id
        ].hex(),
        "valid_from": context.window_start.isoformat(),
        "valid_until": context.window_end.isoformat(),
        "scope": context.required_scope,
        "metadata_origin": "synthetic_certificate",
    }

    return sign_certificate(body, issuer_private_key)


def parse_certificate_time(value):
    """Parse an explicitly timezone-aware certificate timestamp."""

    timestamp = datetime.fromisoformat(value)

    if timestamp.utcoffset() is None:
        raise ValueError("Certificate timezone is missing.")

    return timestamp.astimezone(timezone.utc)


def certificate_checks(
    certificate,
    source_id,
    source_public_key,
    required_scope,
    interval_start,
    interval_end,
):
    """Return PASS, FAIL or UNKNOWN for certificate conditions."""

    results = {
        name: "UNKNOWN"
        for name in CERTIFICATE_CHECKS
    }

    # Missing certificate is not proof of invalidity.
    if certificate is None:
        return results

    if type(certificate) is not dict:
        results["structure"] = "FAIL"
        return results

    body = certificate.get("body")
    signature = certificate.get("signature")

    if body is None:
        return results

    if type(body) is not dict:
        results["structure"] = "FAIL"
        return results

    # Reject malformed fields, but distinguish absent fields.
    if (
        not set(body).issubset(CERTIFICATE_FIELDS)
        or any(type(value) is not str for value in body.values())
    ):
        results["structure"] = "FAIL"
        return results

    if (
        body.get("schema", CERTIFICATE_VERSION)
        != CERTIFICATE_VERSION
        or body.get("metadata_origin", "synthetic_certificate")
        != "synthetic_certificate"
    ):
        results["structure"] = "FAIL"

    elif set(body) == CERTIFICATE_FIELDS:
        results["structure"] = "PASS"

    # Verify issuer authorization independently.
    issuer = body.get("issuer")

    if issuer is None:
        return results

    issuer_public_key = trusted_issuers.get(issuer)

    if issuer_public_key is None:
        results["issuer"] = "FAIL"
        return results

    results["issuer"] = "PASS"

    # Authenticate the certificate before trusting its claims.
    if signature is None:
        return results

    if type(signature) is not bytes or len(signature) != 64:
        results["signature"] = "FAIL"
        return results

    try:
        key = eddsa.import_public_key(issuer_public_key)

        eddsa.new(key, "rfc8032").verify(
            CERTIFICATE_DOMAIN + canonical_bytes(body),
            signature,
        )

        results["signature"] = "PASS"

    except (ValueError, TypeError):
        results["signature"] = "FAIL"
        return results

    # Verify subject identity.
    if "subject" in body and source_id is not None:
        results["subject"] = (
            "PASS" if body["subject"] == source_id else "FAIL"
        )

    # Verify certificate binding to the evidence signing key.
    if "subject_public_key" in body:

        declared_key = body["subject_public_key"]

        if not re.fullmatch(r"[0-9a-f]{64}", declared_key):
            results["key_binding"] = "FAIL"

        elif source_public_key is not None:
            results["key_binding"] = (
                "PASS"
                if declared_key == source_public_key.hex()
                else "FAIL"
            )

    # Verify authorized scope.
    if "scope" in body and required_scope is not None:
        results["scope"] = (
            "PASS"
            if body["scope"] == required_scope
            else "FAIL"
        )

    # Verify validity boundaries and the supplied interval.
    try:
        valid_from = (
            parse_certificate_time(body["valid_from"])
            if "valid_from" in body else None
        )

        valid_until = (
            parse_certificate_time(body["valid_until"])
            if "valid_until" in body else None
        )

    except (ValueError, TypeError):
        results["validity"] = "FAIL"
        return results

    if (
        valid_from is not None
        and valid_until is not None
        and valid_from > valid_until
    ):
        results["validity"] = "FAIL"
        return results

    known_times = [
        timestamp
        for timestamp in (interval_start, interval_end)
        if timestamp is not None
    ]

    # A known violation takes precedence over missing evidence.
    if any(
        (valid_from is not None and timestamp < valid_from)
        or (valid_until is not None and timestamp > valid_until)
        for timestamp in known_times
    ):
        results["validity"] = "FAIL"

    elif (
        valid_from is not None
        and valid_until is not None
        and interval_start is not None
        and interval_end is not None
    ):
        results["validity"] = (
            "PASS" if interval_start <= interval_end else "FAIL"
        )

    return results


# ------------------------------------------------------------
# Issue and validate development certificates
# ------------------------------------------------------------

development_certificates = {
    sequence: build_certificate(context)
    for sequence, context
    in development_policy_contexts.items()
}

valid_certificates = 0

for item in signed_development_evidence:

    evidence = item["evidence"]
    sequence = evidence["sequence"]

    context = development_policy_contexts[sequence]

    results = certificate_checks(
        development_certificates[sequence],
        evidence["source_id"],
        item["public_key"],
        context.required_scope,
        original_measurement_time(
            evidence["records"][0]["timestamp"]
        ),
        datetime.fromisoformat(
            evidence["claims"]["application_timestamp_utc"]
        ),
    )

    assert all(
        result == "PASS" for result in results.values()
    ), results

    valid_certificates += 1


# ------------------------------------------------------------
# Controlled certificate tests
# ------------------------------------------------------------

sample = signed_development_evidence[0]
context = development_policy_contexts[0]
certificate = development_certificates[0]

start = context.window_start
end = context.window_end

boundary_step = timedelta(microseconds=1)


def test_certificate(candidate, start_time=start, end_time=end):
    """Evaluate a certificate against fixed reference conditions."""

    return certificate_checks(
        candidate,
        context.expected_source_id,
        sample["public_key"],
        context.required_scope,
        start_time,
        end_time,
    )


def modified_certificate(**changes):
    """Produce an authentically signed but modified certificate."""

    body = deepcopy(certificate["body"])
    body.update(changes)

    return sign_certificate(body, issuer_private_key)


print("=== CONTROLLED CERTIFICATE TESTS ===")

# 1. Both validity endpoints are inclusive.
assert all(
    value == "PASS"
    for value in test_certificate(certificate).values()
)

print("Valid certificate and exact boundaries: PASS")

# 2. Certificate modified after signing.
tampered = deepcopy(certificate)
tampered["body"]["scope"] = "different_scope"

assert test_certificate(tampered)["signature"] == "FAIL"

print("Certificate tampering detection: PASS")

# 3. Wrong signing key with a trusted issuer identity.
wrong_signer = sign_certificate(
    certificate["body"], untrusted_issuer_key
)

assert test_certificate(wrong_signer)["signature"] == "FAIL"

print("Incorrect issuer signature detection: PASS")

# 4. Internally signed certificate from an untrusted issuer.
untrusted_body = deepcopy(certificate["body"])
untrusted_body["issuer"] = "untrusted_authority"

untrusted_certificate = sign_certificate(
    untrusted_body, untrusted_issuer_key
)

assert test_certificate(untrusted_certificate)["issuer"] == "FAIL"

print("Unauthorized issuer detection: PASS")

# 5. Valid signature, incorrect subject.
assert test_certificate(
    modified_certificate(subject="different_device")
)["subject"] == "FAIL"

print("Subject mismatch detection: PASS")

# 6. Valid signature, incorrect source-key binding.
assert test_certificate(
    modified_certificate(
        subject_public_key=public_key_bytes(
            unauthorized_key
        ).hex()
    )
)["key_binding"] == "FAIL"

print("Public key binding mismatch detection: PASS")

# 7. Valid signature, incorrect authorization scope.
assert test_certificate(
    modified_certificate(scope="different_scope")
)["scope"] == "FAIL"

print("Incorrect scope detection: PASS")

# 8. Expired certificate with an internally valid period.
expired_certificate = modified_certificate(
    valid_from=(start - 2 * boundary_step).isoformat(),
    valid_until=(start - boundary_step).isoformat(),
)

assert test_certificate(
    expired_certificate
)["validity"] == "FAIL"

print("Expired certificate detection: PASS")

# 9. Certificate not yet valid.
assert test_certificate(
    modified_certificate(
        valid_from=(start + boundary_step).isoformat()
    )
)["validity"] == "FAIL"

print("Future validity detection: PASS")

# 10. Reversed certificate validity boundaries.
assert test_certificate(
    modified_certificate(
        valid_from=end.isoformat(),
        valid_until=start.isoformat(),
    )
)["validity"] == "FAIL"

print("Reversed validity period detection: PASS")

# 11. Missing certificate.
assert all(
    value == "UNKNOWN"
    for value in test_certificate(None).values()
)

print("Missing certificate handling: PASS")

# 12. Missing certificate signature.
unsigned_certificate = {
    "body": deepcopy(certificate["body"])
}

assert test_certificate(
    unsigned_certificate
)["signature"] == "UNKNOWN"

print("Missing signature handling: PASS")

# 13. Missing temporal evidence.
assert test_certificate(
    certificate, end_time=None
)["validity"] == "UNKNOWN"

print("Missing temporal evidence handling: PASS")

# 14. Proven violation takes precedence over missing time.
assert test_certificate(
    expired_certificate, end_time=None
)["validity"] == "FAIL"

print("Violation precedence: PASS")

# ------------------------------------------------------------
# Report results
# ------------------------------------------------------------

print("\n=== CERTIFICATE CONFIGURATION ===")
print("Certificate version:", CERTIFICATE_VERSION)
print("Trusted issuers:", len(trusted_issuers))
print("Independent issuer key: PASS")

print("\n=== DEVELOPMENT VALIDATION ===")
print("Certificates issued:", len(development_certificates))
print("Fully valid certificates:", valid_certificates)
print("Controlled test categories: 14/14 PASS")

assert valid_certificates == DEVELOPMENT_COUNT

print("\nVALIDATION: PASS")

=== CONTROLLED CERTIFICATE TESTS ===
Valid certificate and exact boundaries: PASS
Certificate tampering detection: PASS
Incorrect issuer signature detection: PASS
Unauthorized issuer detection: PASS
Subject mismatch detection: PASS
Public key binding mismatch detection: PASS
Incorrect scope detection: PASS
Expired certificate detection: PASS
Future validity detection: PASS
Reversed validity period detection: PASS
Missing certificate handling: PASS
Missing signature handling: PASS
Missing temporal evidence handling: PASS
Violation precedence: PASS

=== CERTIFICATE CONFIGURATION ===
Certificate version: iot-authorization-certificate-v1
Trusted issuers: 1
Independent issuer key: PASS

=== DEVELOPMENT VALIDATION ===
Certificates issued: 65
Fully valid certificates: 65
Controlled test categories: 14/14 PASS

VALIDATION: PASS


In [12]:
# ============================================================
# CELL 10 — Common evidence-validation engine
# ============================================================
# Implement R06-R10 using the approved independent policy.
#
# Reference: PROTOCOL_AMENDMENT_02.md, Sections 2-6.
#
# Do not use scenario labels, reference outcomes or
# unsupported physical measurement thresholds.

from decimal import Decimal, InvalidOperation

PASS = "PASS"
FAIL = "FAIL"
UNKNOWN = "UNKNOWN"

ACCEPT = "ACCEPT"
REJECT = "REJECT"
INDETERMINATE = "INDETERMINATE"

RECORD_FIELDS = {
    "timestamp",
    "voltage",
    "current",
    "power",
    "energy_Wh",
    "cumulative_energy_Wh",
    "device_id",
    "chain_hash",
}

NUMERIC_FIELDS = tuple(
    field for field, unit in POLICY_UNITS
)

EVIDENCE_FIELDS = {
    "schema",
    "sequence",
    "source_id",
    "previous_commitment",
    "claims",
    "records",
}

CLAIM_FIELDS = {
    "application_timestamp_utc",
    "metadata_origin",
    "measurement_units",
}

POLICY_CHECKS = (
    "R06.record_count",
    "R07.sequence",
    "R07.chronology",
    "R07.application_time",
    "R07.time_window",
    "R08.evidence_structure",
    "R08.record_structure",
    "R08.numeric_values",
    "R08.units",
    "R08.source_consistency",
    "R08.duplicates",
)


def aggregate_decision(checks):
    """Apply the approved three-class decision precedence."""

    values = list(checks.values())

    if any(value not in (PASS, FAIL, UNKNOWN) for value in values):
        raise ValueError("Invalid verification status.")

    if FAIL in values:
        return REJECT

    if not values or UNKNOWN in values:
        return INDETERMINATE

    return ACCEPT


def check_status(condition):
    """Convert a Boolean condition into a verification status."""

    return PASS if condition else FAIL


def valid_digest(value):
    """Check the representation of a SHA-256 digest."""

    return (
        type(value) is str
        and re.fullmatch(r"[0-9a-f]{64}", value) is not None
    )


def finite_numeric(value):
    """Check numerical representation without physical thresholds."""

    if type(value) is not str:
        return False

    try:
        return Decimal(value).is_finite()

    except (InvalidOperation, ValueError):
        return False


def parse_measurement_timestamp(value):
    """Parse the original logger's timezone-naive UTC format."""

    if type(value) is not str or "T" not in value:
        return None

    try:
        timestamp = datetime.fromisoformat(value)

        if timestamp.tzinfo is not None:
            return None

        return timestamp.replace(tzinfo=timezone.utc)

    except ValueError:
        return None


def parse_application_timestamp(value):
    """Require an explicitly UTC application timestamp."""

    if type(value) is not str or "T" not in value:
        return None

    try:
        timestamp = datetime.fromisoformat(value)

        if timestamp.utcoffset() != timedelta(0):
            return None

        return timestamp.astimezone(timezone.utc)

    except ValueError:
        return None


def evaluate_evidence_policy(
    evidence,
    source_public_key,
    certificate,
    context,
):
    """Evaluate evidence independently of its infrastructure."""

    checks = {
        name: UNKNOWN
        for name in POLICY_CHECKS
    }

    source_id = None
    application_time = None
    known_times = []
    complete_times = False

    # --------------------------------------------------------
    # R08: Evidence structure
    # --------------------------------------------------------

    if evidence is not None and type(evidence) is not dict:
        checks["R08.evidence_structure"] = FAIL

    if type(evidence) is dict:

        source = evidence.get("source_id")
        sequence = evidence.get("sequence")
        previous = evidence.get("previous_commitment")

        claims = evidence.get("claims")
        records = evidence.get("records")

        if type(source) is str and source:
            source_id = source

        valid_structure = (
            set(evidence) == EVIDENCE_FIELDS
            and evidence.get("schema") == SCHEMA_VERSION
            and type(sequence) is int
            and sequence >= 0
            and source_id is not None
            and (previous is None or valid_digest(previous))
            and type(claims) is dict
            and set(claims) == CLAIM_FIELDS
            and type(records) is list
        )

        if valid_structure:
            valid_structure = (
                type(claims["metadata_origin"]) is str
                and bool(claims["metadata_origin"])
            )

        checks["R08.evidence_structure"] = check_status(
            valid_structure
        )

        # ----------------------------------------------------
        # R07: Expected sequence
        # ----------------------------------------------------

        if type(sequence) is int and context is not None:

            checks["R07.sequence"] = check_status(
                sequence == context.expected_sequence
            )

        # ----------------------------------------------------
        # R08: Declared units and application timestamp
        # ----------------------------------------------------

        if type(claims) is dict:

            units = claims.get("measurement_units")

            if units is not None:

                if type(units) is not dict:
                    checks["R08.units"] = FAIL

                elif context is not None:

                    checks["R08.units"] = check_status(
                        units == dict(context.expected_units)
                    )

            application_time = parse_application_timestamp(
                claims.get("application_timestamp_utc")
            )

            if application_time is None:
                checks["R07.application_time"] = FAIL

        # ----------------------------------------------------
        # R06: Record completeness
        # ----------------------------------------------------

        if type(records) is list:

            if context is not None:

                checks["R06.record_count"] = check_status(
                    len(records) == context.expected_record_count
                )

            # ------------------------------------------------
            # R08: Record structure and numerical validity
            # ------------------------------------------------

            if records:

                structure_valid = all(
                    type(record) is dict
                    and set(record) == RECORD_FIELDS
                    and all(
                        type(value) is str
                        for value in record.values()
                    )
                    and bool(record["device_id"])
                    and valid_digest(record["chain_hash"])
                    for record in records
                )

                checks["R08.record_structure"] = check_status(
                    structure_valid
                )

                dictionary_records = [
                    record for record in records
                    if type(record) is dict
                ]

                checks["R08.numeric_values"] = check_status(
                    len(dictionary_records) == len(records)
                    and all(
                        finite_numeric(record.get(field))
                        for record in dictionary_records
                        for field in NUMERIC_FIELDS
                    )
                )

                # --------------------------------------------
                # R07: Chronology
                # --------------------------------------------

                timestamps = [
                    parse_measurement_timestamp(
                        record.get("timestamp")
                    )
                    if type(record) is dict else None
                    for record in records
                ]

                known_times = [
                    timestamp for timestamp in timestamps
                    if timestamp is not None
                ]

                complete_times = (
                    len(known_times) == len(records)
                )

                checks["R07.chronology"] = check_status(
                    complete_times
                    and all(
                        current > previous
                        for previous, current in zip(
                            known_times, known_times[1:]
                        )
                    )
                )

                # Application time must not precede
                # any available measurement.
                if application_time is not None:

                    if any(
                        timestamp > application_time
                        for timestamp in known_times
                    ):
                        checks["R07.application_time"] = FAIL

                    elif complete_times:
                        checks["R07.application_time"] = PASS

                # --------------------------------------------
                # R08: Source consistency
                # --------------------------------------------

                record_sources = [
                    record.get("device_id")
                    for record in dictionary_records
                ]

                if source_id is not None:

                    if any(
                        record_source != source_id
                        for record_source in record_sources
                    ):
                        checks["R08.source_consistency"] = FAIL

                    elif context is not None:

                        checks["R08.source_consistency"] = (
                            check_status(
                                source_id
                                == context.expected_source_id
                                and structure_valid
                            )
                        )

                # --------------------------------------------
                # R08: Duplicate measurement identifiers
                # --------------------------------------------

                # Use device and parsed measurement time.
                # Also check duplicated original chain hashes.
                identifiers = []
                record_hashes = []

                for record, timestamp in zip(records, timestamps):

                    if type(record) is not dict:
                        continue

                    device = record.get("device_id")
                    digest = record.get("chain_hash")

                    if type(device) is str and timestamp is not None:
                        identifiers.append((device, timestamp))

                    if valid_digest(digest):
                        record_hashes.append(digest)

                duplicate_found = (
                    len(identifiers) != len(set(identifiers))
                    or len(record_hashes) != len(set(record_hashes))
                )

                if duplicate_found:
                    checks["R08.duplicates"] = FAIL

                elif (
                    len(identifiers) == len(records)
                    and len(record_hashes) == len(records)
                ):
                    checks["R08.duplicates"] = PASS

        # ----------------------------------------------------
        # R07: Independent temporal window
        # ----------------------------------------------------

        observed_times = list(known_times)

        if application_time is not None:
            observed_times.append(application_time)

        if context is not None and observed_times:

            outside_window = any(
                not (
                    context.window_start
                    <= timestamp
                    <= context.window_end
                )
                for timestamp in observed_times
            )

            if outside_window:
                checks["R07.time_window"] = FAIL

            elif complete_times and application_time is not None:
                checks["R07.time_window"] = PASS

    # --------------------------------------------------------
    # R09: Institutional certificate validity
    # --------------------------------------------------------

    certificate_times = list(known_times)

    if application_time is not None:
        certificate_times.append(application_time)

    interval_start = (
        min(certificate_times)
        if certificate_times else None
    )

    interval_end = (
        max(certificate_times)
        if certificate_times else None
    )

    usable_public_key = (
        source_public_key
        if type(source_public_key) is bytes
        and len(source_public_key) == 32
        else None
    )

    institutional = certificate_checks(
        certificate,
        source_id,
        usable_public_key,
        context.required_scope if context is not None else None,
        interval_start,
        interval_end,
    )

    # A partial temporal interval cannot establish full
    # certificate validity, but it can demonstrate a violation.
    if (
        institutional["validity"] == PASS
        and not (
            complete_times and application_time is not None
        )
    ):
        institutional["validity"] = UNKNOWN

    checks.update({
        f"R09.{name}": status
        for name, status in institutional.items()
    })

    # --------------------------------------------------------
    # R10: Three-class policy decision
    # --------------------------------------------------------

    return {
        "decision": aggregate_decision(checks),
        "checks": checks,
        "violations": [
            name for name, status in checks.items()
            if status == FAIL
        ],
        "unresolved": [
            name for name, status in checks.items()
            if status == UNKNOWN
        ],
    }


# ------------------------------------------------------------
# Validate all unmodified development controls
# ------------------------------------------------------------

development_policy_results = []

for item in signed_development_evidence:

    evidence = item["evidence"]
    sequence = evidence["sequence"]

    result = evaluate_evidence_policy(
        evidence,
        item["public_key"],
        development_certificates[sequence],
        development_policy_contexts[sequence],
    )

    assert result["decision"] == ACCEPT, result
    assert not result["violations"]
    assert not result["unresolved"]

    development_policy_results.append(result)


# ------------------------------------------------------------
# Controlled policy tests
# ------------------------------------------------------------

sample = signed_development_evidence[0]

original = sample["evidence"]
context = development_policy_contexts[0]
certificate = development_certificates[0]
public_key = sample["public_key"]


def check_policy_test(name, result, expected, condition=None):

    assert result["decision"] == expected, (
        name, result
    )

    if condition is not None:
        assert condition in result["violations"], result

    print(f"{name}: PASS")


def evaluate_sample(
    evidence=original,
    supporting_certificate=certificate,
    policy_context=context,
):

    return evaluate_evidence_policy(
        evidence,
        public_key,
        supporting_certificate,
        policy_context,
    )


print("=== CONTROLLED EVIDENCE-POLICY TESTS ===")

# 1. Unmodified evidence.
check_policy_test(
    "Valid control",
    evaluate_sample(),
    ACCEPT,
)

# 2. Omitted record from an available batch.
modified = deepcopy(original)
modified["records"].pop()

check_policy_test(
    "Record omission",
    evaluate_sample(modified),
    REJECT,
    "R06.record_count",
)

# 3. Incorrect expected sequence.
modified = deepcopy(original)
modified["sequence"] += 1

check_policy_test(
    "Incorrect sequence",
    evaluate_sample(modified),
    REJECT,
    "R07.sequence",
)

# 4. Application timestamp precedes measurements.
modified = deepcopy(original)

modified["claims"]["application_timestamp_utc"] = (
    context.window_start - timedelta(microseconds=1)
).isoformat()

check_policy_test(
    "Temporal inconsistency",
    evaluate_sample(modified),
    REJECT,
    "R07.application_time",
)

# 5. Application timestamp outside permitted window.
modified = deepcopy(original)

modified["claims"]["application_timestamp_utc"] = (
    context.window_end + timedelta(microseconds=1)
).isoformat()

check_policy_test(
    "Freshness window violation",
    evaluate_sample(modified),
    REJECT,
    "R07.time_window",
)

# 6. Incorrect record ordering.
modified = deepcopy(original)
modified["records"].reverse()

check_policy_test(
    "Chronological violation",
    evaluate_sample(modified),
    REJECT,
    "R07.chronology",
)

# 7. Incorrect declared measurement unit.
modified = deepcopy(original)
modified["claims"]["measurement_units"]["current"] = "mA"

check_policy_test(
    "Incorrect measurement unit",
    evaluate_sample(modified),
    REJECT,
    "R08.units",
)

# 8. Non-finite numerical measurement.
modified = deepcopy(original)
modified["records"][0]["voltage"] = "NaN"

check_policy_test(
    "Non-finite numerical value",
    evaluate_sample(modified),
    REJECT,
    "R08.numeric_values",
)

# 9. Missing required field in a presented record.
modified = deepcopy(original)
del modified["records"][0]["voltage"]

check_policy_test(
    "Invalid record structure",
    evaluate_sample(modified),
    REJECT,
    "R08.record_structure",
)

# 10. Duplicate measurement without changing record count.
modified = deepcopy(original)

modified["records"][1] = deepcopy(
    modified["records"][0]
)

check_policy_test(
    "Duplicate measurement",
    evaluate_sample(modified),
    REJECT,
    "R08.duplicates",
)

# 11. Inconsistent record source.
modified = deepcopy(original)
modified["records"][0]["device_id"] = "different_device"

check_policy_test(
    "Source inconsistency",
    evaluate_sample(modified),
    REJECT,
    "R08.source_consistency",
)

# 12. Expired institutional certificate.
check_policy_test(
    "Expired certificate",
    evaluate_sample(
        supporting_certificate=expired_certificate
    ),
    REJECT,
    "R09.validity",
)

# 13. Missing certificate.
check_policy_test(
    "Missing certificate",
    evaluate_sample(supporting_certificate=None),
    INDETERMINATE,
)

# 14. Entire evidence payload unavailable.
check_policy_test(
    "Unavailable evidence",
    evaluate_sample(
        evidence=None,
        supporting_certificate=None,
    ),
    INDETERMINATE,
)

# 15. Independent policy context unavailable.
check_policy_test(
    "Unavailable policy context",
    evaluate_sample(policy_context=None),
    INDETERMINATE,
)

# 16. Proven violation combined with missing evidence.
modified = deepcopy(original)
modified["claims"]["measurement_units"]["current"] = "mA"

check_policy_test(
    "Violation precedence",
    evaluate_sample(
        modified,
        supporting_certificate=None,
    ),
    REJECT,
    "R08.units",
)

# 17. Valid numerical representation with an unusual value.
# No unsupported physical threshold or original-value
# comparison is applied by this policy engine.
modified = deepcopy(original)
modified["records"][0]["current"] = "-0.125"

check_policy_test(
    "Unusual finite measurement",
    evaluate_sample(modified),
    ACCEPT,
)


# ------------------------------------------------------------
# Independent decision aggregation tests
# ------------------------------------------------------------

assert aggregate_decision({"a": PASS}) == ACCEPT

assert aggregate_decision({
    "a": PASS, "b": UNKNOWN
}) == INDETERMINATE

assert aggregate_decision({
    "a": FAIL, "b": UNKNOWN
}) == REJECT

assert aggregate_decision({}) == INDETERMINATE


# ------------------------------------------------------------
# Report results
# ------------------------------------------------------------

print("\n=== DEVELOPMENT POLICY VALIDATION ===")
print("Policy conditions per batch:",
      len(development_policy_results[0]["checks"]))

print("Development controls:",
      len(development_policy_results))

print("Accepted:", sum(
    result["decision"] == ACCEPT
    for result in development_policy_results
))

print("Rejected:", sum(
    result["decision"] == REJECT
    for result in development_policy_results
))

print("Indeterminate:", sum(
    result["decision"] == INDETERMINATE
    for result in development_policy_results
))

print("\n=== IMPLEMENTATION VALIDATION ===")
print("Controlled policy tests: 17/17 PASS")
print("Decision aggregation tests: 4/4 PASS")
print("Independent policy enforcement: PASS")
print("Physical accuracy thresholds: NONE")

print("\nVALIDATION: PASS")

=== CONTROLLED EVIDENCE-POLICY TESTS ===
Valid control: PASS
Record omission: PASS
Incorrect sequence: PASS
Temporal inconsistency: PASS
Freshness window violation: PASS
Chronological violation: PASS
Incorrect measurement unit: PASS
Non-finite numerical value: PASS
Invalid record structure: PASS
Duplicate measurement: PASS
Source inconsistency: PASS
Expired certificate: PASS
Missing certificate: PASS
Unavailable evidence: PASS
Unavailable policy context: PASS
Violation precedence: PASS
Unusual finite measurement: PASS

=== DEVELOPMENT POLICY VALIDATION ===
Policy conditions per batch: 18
Development controls: 65
Accepted: 65
Rejected: 0
Indeterminate: 0

=== IMPLEMENTATION VALIDATION ===
Controlled policy tests: 17/17 PASS
Decision aggregation tests: 4/4 PASS
Independent policy enforcement: PASS
Physical accuracy thresholds: NONE

VALIDATION: PASS


In [13]:
# ============================================================
# CELL 11 — Common cryptographic verification
# ============================================================
# Implement verification conditions shared by all architectures.
#
# References:
# REQUIREMENTS.md, R01, R02 and R04.
# PROTOCOL_ADDENDUM.md, Sections 1-3.
#
# Keep authentication, authorization and history separate.
# No hidden scenario labels or expected outcomes are inputs.


COMMON_CHECKS = (
    "R01.commitment",
    "R02.signature",
    "R02.authorization",
    "R04.sequence",
    "R04.predecessor",
)


@dataclass(frozen=True)
class VerifierHistory:
    """Previously retained verifier state."""

    next_sequence: int
    previous_commitment: str | None

    def __post_init__(self):

        if (
            type(self.next_sequence) is not int
            or self.next_sequence < 0
        ):
            raise ValueError("Invalid retained sequence.")

        if self.next_sequence == 0:

            if self.previous_commitment is not None:
                raise ValueError("Invalid genesis state.")

        elif not valid_digest(self.previous_commitment):

            raise ValueError("Invalid retained commitment.")


def verify_common_crypto(envelope, history):
    """Evaluate shared cryptographic and history conditions."""

    checks = {
        name: UNKNOWN
        for name in COMMON_CHECKS
    }

    computed_commitment = None

    if envelope is None:

        return {
            "checks": checks,
            "computed_commitment": computed_commitment,
        }

    if type(envelope) is not dict:

        checks["R01.commitment"] = FAIL

        return {
            "checks": checks,
            "computed_commitment": computed_commitment,
        }

    evidence = envelope.get("evidence")
    declared_commitment = envelope.get("commitment")
    signature = envelope.get("signature")
    public_key = envelope.get("public_key")

    # --------------------------------------------------------
    # R01: Canonical commitment consistency
    # --------------------------------------------------------

    if evidence is not None:

        if type(evidence) is not dict:

            checks["R01.commitment"] = FAIL

        else:

            try:
                computed_commitment = evidence_commitment(
                    evidence
                )

            except (ValueError, TypeError, UnicodeError):

                checks["R01.commitment"] = FAIL

            else:

                # The commitment can be computed from the
                # evidence. A separate declared digest is optional.
                checks["R01.commitment"] = check_status(
                    declared_commitment is None
                    or declared_commitment == computed_commitment
                )

    # --------------------------------------------------------
    # R02: Cryptographic signature verification
    # --------------------------------------------------------

    key_valid = (
        type(public_key) is bytes
        and len(public_key) == 32
    )

    signature_valid = (
        type(signature) is bytes
        and len(signature) == 64
    )

    if (
        (public_key is not None and not key_valid)
        or (signature is not None and not signature_valid)
    ):

        checks["R02.signature"] = FAIL

    elif (
        computed_commitment is not None
        and key_valid
        and signature_valid
    ):

        checks["R02.signature"] = check_status(
            verify_signature(
                evidence,
                signature,
                public_key,
            )
        )

    # --------------------------------------------------------
    # R02: Independent source authorization
    # --------------------------------------------------------

    if type(evidence) is dict:

        source_id = evidence.get("source_id")

        if (
            source_id is not None
            and (type(source_id) is not str or not source_id)
        ):

            checks["R02.authorization"] = FAIL

        elif type(source_id) is str and key_valid:

            checks["R02.authorization"] = check_status(
                is_authorized(evidence, public_key)
            )

    # --------------------------------------------------------
    # R04: Retained history verification
    # --------------------------------------------------------

    if type(evidence) is dict:

        sequence = evidence.get("sequence")

        if sequence is not None:

            if type(sequence) is not int or sequence < 0:

                checks["R04.sequence"] = FAIL

            elif history is not None:

                checks["R04.sequence"] = check_status(
                    sequence == history.next_sequence
                )

        if "previous_commitment" in evidence:

            previous = evidence["previous_commitment"]

            if previous is not None and not valid_digest(previous):

                checks["R04.predecessor"] = FAIL

            elif history is not None:

                checks["R04.predecessor"] = check_status(
                    previous == history.previous_commitment
                )

    return {
        "checks": checks,
        "computed_commitment": computed_commitment,
    }


# ------------------------------------------------------------
# Validate the chronological development history
# ------------------------------------------------------------

development_histories = {}
development_common_results = []

history = VerifierHistory(
    next_sequence=0,
    previous_commitment=None,
)

for item in signed_development_evidence:

    evidence = item["evidence"]

    # Preserve the state available before verification.
    development_histories[history.next_sequence] = history

    result = verify_common_crypto(item, history)

    assert aggregate_decision(result["checks"]) == ACCEPT

    assert result["computed_commitment"] == item["commitment"]

    development_common_results.append(result)

    # Advance only after successful common verification.
    history = VerifierHistory(
        next_sequence=history.next_sequence + 1,
        previous_commitment=result["computed_commitment"],
    )

assert history.next_sequence == DEVELOPMENT_COUNT

assert history.previous_commitment == (
    development_evidence[-1]["commitment"]
)


# ------------------------------------------------------------
# Controlled common-verification tests
# ------------------------------------------------------------

sample = signed_development_evidence[0]
initial_history = development_histories[0]


def check_common_test(
    name, envelope, retained_history,
    expected, condition=None, status=None
):

    result = verify_common_crypto(
        envelope, retained_history
    )

    assert aggregate_decision(
        result["checks"]
    ) == expected, (name, result)

    if condition is not None:

        assert result["checks"][condition] == status, result

    print(f"{name}: PASS")


def resign_envelope(envelope, private_key):
    """Create a newly signed experimental evidence envelope."""

    evidence = envelope["evidence"]

    return {
        "evidence": deepcopy(evidence),
        "commitment": evidence_commitment(evidence),
        "public_key": public_key_bytes(private_key),
        "signature": sign_evidence(evidence, private_key),
    }


print("=== COMMON CRYPTOGRAPHIC TESTS ===")

# 1. Valid evidence.
check_common_test(
    "Valid signed evidence",
    sample, initial_history, ACCEPT,
)

# 2. Modification after signing.
modified = deepcopy(sample)
modified["evidence"]["records"][0]["voltage"] = "999"

check_common_test(
    "Post-signing modification",
    modified, initial_history, REJECT,
    "R02.signature", FAIL,
)

# 3. Incorrect declared commitment.
modified = deepcopy(sample)
modified["commitment"] = flip_digest_bit(
    bytes.fromhex(modified["commitment"])
).hex()

check_common_test(
    "Commitment mismatch",
    modified, initial_history, REJECT,
    "R01.commitment", FAIL,
)

# 4. Altered signature.
modified = deepcopy(sample)
modified["signature"] = flip_digest_bit(
    modified["signature"]
)

check_common_test(
    "Invalid signature",
    modified, initial_history, REJECT,
    "R02.signature", FAIL,
)

# 5. Correct signature from an unauthorized key.
modified = resign_envelope(
    sample, unauthorized_key
)

assert verify_common_crypto(
    modified, initial_history
)["checks"]["R02.signature"] == PASS

check_common_test(
    "Unauthorized signing key",
    modified, initial_history, REJECT,
    "R02.authorization", FAIL,
)

# 6. Missing signature.
modified = deepcopy(sample)
del modified["signature"]

check_common_test(
    "Missing signature",
    modified, initial_history, INDETERMINATE,
    "R02.signature", UNKNOWN,
)

# 7. Missing public key.
modified = deepcopy(sample)
del modified["public_key"]

check_common_test(
    "Missing public key",
    modified, initial_history, INDETERMINATE,
    "R02.signature", UNKNOWN,
)

# 8. Missing retained history.
check_common_test(
    "Unavailable retained history",
    sample, None, INDETERMINATE,
    "R04.sequence", UNKNOWN,
)

# 9. Replay against a verifier that has advanced.
check_common_test(
    "Replay against retained history",
    sample, development_histories[1], REJECT,
    "R04.sequence", FAIL,
)

# 10. Correctly signed but incorrect predecessor.
modified = deepcopy(signed_development_evidence[1])

modified["evidence"]["previous_commitment"] = (
    bytes(32).hex()
)

modified = resign_envelope(
    modified,
    source_private_keys[
        modified["evidence"]["source_id"]
    ],
)

check_common_test(
    "Incorrect signed predecessor",
    modified, development_histories[1], REJECT,
    "R04.predecessor", FAIL,
)

# 11. Entire evidence envelope unavailable.
check_common_test(
    "Unavailable evidence",
    None, initial_history, INDETERMINATE,
)

# 12. Correctly signed evidence with an incorrect unit.
modified = deepcopy(sample)

modified["evidence"]["claims"][
    "measurement_units"
]["current"] = "mA"

modified = resign_envelope(
    modified,
    source_private_keys[
        modified["evidence"]["source_id"]
    ],
)

check_common_test(
    "Cryptographically valid incorrect unit",
    modified, initial_history, ACCEPT,
)

# Confirm that the independent policy detects the violation.
policy_result = evaluate_evidence_policy(
    modified["evidence"],
    modified["public_key"],
    development_certificates[0],
    development_policy_contexts[0],
)

assert policy_result["decision"] == REJECT
assert "R08.units" in policy_result["violations"]

# 13. Proven violation with unavailable history.
modified = deepcopy(sample)
modified["signature"] = flip_digest_bit(
    modified["signature"]
)

check_common_test(
    "Violation with missing history",
    modified, None, REJECT,
    "R02.signature", FAIL,
)

# 14. Optional separately declared commitment absent.
modified = deepcopy(sample)
del modified["commitment"]

check_common_test(
    "Computed commitment without declared digest",
    modified, initial_history, ACCEPT,
    "R01.commitment", PASS,
)


# ------------------------------------------------------------
# Report validation results
# ------------------------------------------------------------

print("\n=== DEVELOPMENT VERIFICATION ===")
print("Common conditions per batch:", len(COMMON_CHECKS))
print("Development controls:", len(development_common_results))
print("Retained history snapshots:", len(development_histories))

print("Accepted:", sum(
    aggregate_decision(result["checks"]) == ACCEPT
    for result in development_common_results
))

print("Rejected:", sum(
    aggregate_decision(result["checks"]) == REJECT
    for result in development_common_results
))

print("Indeterminate:", sum(
    aggregate_decision(result["checks"]) == INDETERMINATE
    for result in development_common_results
))

print("\n=== FINAL HISTORY ===")
print("Next expected sequence:", history.next_sequence)
print("Final retained commitment:", history.previous_commitment)

print("\n=== IMPLEMENTATION VALIDATION ===")
print("Controlled tests: 14/14 PASS")
print("Authentication and authorization separation: PASS")
print("Cryptographic and semantic separation: PASS")
print("Independent retained history: PASS")

print("\nVALIDATION: PASS")

=== COMMON CRYPTOGRAPHIC TESTS ===
Valid signed evidence: PASS
Post-signing modification: PASS
Commitment mismatch: PASS
Invalid signature: PASS
Unauthorized signing key: PASS
Missing signature: PASS
Missing public key: PASS
Unavailable retained history: PASS
Replay against retained history: PASS
Incorrect signed predecessor: PASS
Unavailable evidence: PASS
Cryptographically valid incorrect unit: PASS
Violation with missing history: PASS
Computed commitment without declared digest: PASS

=== DEVELOPMENT VERIFICATION ===
Common conditions per batch: 5
Development controls: 65
Retained history snapshots: 65
Accepted: 65
Rejected: 0
Indeterminate: 0

=== FINAL HISTORY ===
Next expected sequence: 65
Final retained commitment: b95e58c148f97265dfef56a9fd718d3e38e357a458d629112d443a6f1638f2ec

=== IMPLEMENTATION VALIDATION ===
Controlled tests: 14/14 PASS
Authentication and authorization separation: PASS
Cryptographic and semantic separation: PASS
Independent retained history: PASS

VALIDATIO

In [14]:
# ============================================================
# CELL 12 — Transparency-log infrastructure verification
# ============================================================
# Integrate Merkle inclusion, checkpoint authentication,
# append-only consistency and witness quorum verification.
#
# References:
# REQUIREMENTS.md, R03-R05.
# RFC 9162, Sections 2.1.3 and 2.1.4.
# https://doi.org/10.17487/RFC9162
#
# The verifier receives submitted supporting evidence and
# previously retained state, not hidden scenario information.

LOG_CHECKS = (
    "R03.log_inclusion",
    "R04.log_operator",
    "R04.log_consistency",
    "R05.witness_quorum",
)


@dataclass(frozen=True)
class RetainedLogCheckpoint:
    """Previously accepted transparency-log state."""

    size: int
    root: str

    def __post_init__(self):

        if (
            type(self.size) is not int
            or self.size < 0
            or not valid_digest(self.root)
        ):
            raise ValueError("Invalid retained checkpoint.")

        if self.size == 0 and self.root != log_tree_root([]).hex():
            raise ValueError("Invalid empty checkpoint.")


def build_log_support(entries, index, bundle, old_size):
    """Construct supporting evidence for an experimental submission."""

    checkpoint = bundle["checkpoint"]

    assert checkpoint == make_checkpoint(entries)

    return {
        "checkpoint": deepcopy(checkpoint),
        "operator_signature": bundle["operator_signature"],
        "cosignatures": deepcopy(bundle["cosignatures"]),
        "inclusion_proof": log_inclusion_proof(entries, index),
        "consistency_proof": log_consistency_proof(
            entries, old_size
        ),
    }


def valid_proof_structure(proof):
    """Check the representation of a submitted Merkle path."""

    return (
        type(proof) in (list, tuple)
        and all(
            type(node) is bytes and len(node) == 32
            for node in proof
        )
    )


def verify_log_support(
    computed_commitment,
    sequence,
    support,
    retained_checkpoint,
):
    """Verify transparency-log supporting evidence."""

    checks = {
        name: UNKNOWN
        for name in LOG_CHECKS
    }

    valid_votes = set()
    invalid_votes = 0

    def result():

        return {
            "checks": checks,
            "valid_witnesses": len(valid_votes),
            "invalid_witness_votes": invalid_votes,
        }

    if support is None:
        return result()

    if type(support) is not dict:

        checks["R04.log_operator"] = FAIL
        return result()

    checkpoint = support.get("checkpoint")
    operator_signature = support.get("operator_signature")

    inclusion = support.get("inclusion_proof")
    consistency = support.get("consistency_proof")
    cosignatures = support.get("cosignatures")

    # Malformed supplied proof paths are distinguishable
    # from paths that are simply unavailable.
    if inclusion is not None and not valid_proof_structure(inclusion):
        checks["R03.log_inclusion"] = FAIL

    if consistency is not None and not valid_proof_structure(consistency):
        checks["R04.log_consistency"] = FAIL

    if checkpoint is None:
        return result()

    if not valid_checkpoint(checkpoint):

        checks["R04.log_operator"] = FAIL
        return result()

    size = checkpoint["size"]
    root = bytes.fromhex(checkpoint["root"])

    # --------------------------------------------------------
    # R04: Authenticate the checkpoint operator
    # --------------------------------------------------------

    if operator_signature is not None:

        checks["R04.log_operator"] = check_status(
            type(operator_signature) is bytes
            and len(operator_signature) == 64
            and verify_checkpoint_signature(
                checkpoint,
                operator_signature,
                operator_public_key,
                OPERATOR_DOMAIN,
            )
        )

    # --------------------------------------------------------
    # R05: Verify distinct authorized witness signatures
    # --------------------------------------------------------

    if cosignatures is not None:

        if type(cosignatures) not in (list, tuple):

            checks["R05.witness_quorum"] = FAIL

        else:

            for vote in cosignatures:

                if (
                    type(vote) not in (list, tuple)
                    or len(vote) != 2
                ):
                    invalid_votes += 1
                    continue

                name, signature = vote

                if (
                    type(name) is not str
                    or name not in witness_public_keys
                    or type(signature) is not bytes
                    or len(signature) != 64
                ):
                    invalid_votes += 1
                    continue

                if verify_checkpoint_signature(
                    checkpoint,
                    signature,
                    witness_public_keys[name],
                    WITNESS_DOMAIN,
                ):

                    valid_votes.add(name)

                else:
                    invalid_votes += 1

            # Insufficient valid votes do not prove equivocation.
            checks["R05.witness_quorum"] = (
                PASS
                if len(valid_votes) >= WITNESS_QUORUM
                else UNKNOWN
            )

    # --------------------------------------------------------
    # R03: Inclusion of the submitted evidence commitment
    # --------------------------------------------------------

    if (
        valid_digest(computed_commitment)
        and type(sequence) is int
        and inclusion is not None
        and valid_proof_structure(inclusion)
    ):

        checks["R03.log_inclusion"] = check_status(
            verify_log_inclusion(
                bytes.fromhex(computed_commitment),
                sequence,
                size,
                inclusion,
                root,
            )
        )

    # --------------------------------------------------------
    # R04: Consistency against previously retained state
    # --------------------------------------------------------

    if (
        retained_checkpoint is not None
        and checks["R04.log_consistency"] != FAIL
    ):

        old_size = retained_checkpoint.size
        old_root = bytes.fromhex(retained_checkpoint.root)

        # An authenticated conflicting or regressive
        # checkpoint demonstrates a history violation.
        if (
            checks["R04.log_operator"] == PASS
            and (
                size < old_size
                or (size == old_size and root != old_root)
            )
        ):

            checks["R04.log_consistency"] = FAIL

        elif consistency is not None:

            checks["R04.log_consistency"] = check_status(
                verify_log_consistency(
                    old_size,
                    size,
                    old_root,
                    root,
                    consistency,
                )
            )

        elif old_size == 0 or (
            size == old_size and root == old_root
        ):

            # Empty and unchanged histories need no
            # additional consistency path.
            checks["R04.log_consistency"] = check_status(
                verify_log_consistency(
                    old_size,
                    size,
                    old_root,
                    root,
                    [],
                )
            )

    return result()


# ------------------------------------------------------------
# Validate all development checkpoints chronologically
# ------------------------------------------------------------

development_log_supports = {}
development_log_histories = {}
development_log_results = []

retained = RetainedLogCheckpoint(
    size=0,
    root=log_tree_root([]).hex(),
)

for index, item in enumerate(signed_development_evidence):

    entries = log_entries[:index + 1]
    bundle = witnessed_checkpoints[index]

    support = build_log_support(
        entries,
        index,
        bundle,
        retained.size,
    )

    common = verify_common_crypto(
        item,
        development_histories[index],
    )

    result = verify_log_support(
        common["computed_commitment"],
        item["evidence"]["sequence"],
        support,
        retained,
    )

    assert aggregate_decision(result["checks"]) == ACCEPT, result
    assert result["valid_witnesses"] == WITNESS_COUNT

    development_log_supports[index] = support
    development_log_histories[index] = retained
    development_log_results.append(result)

    # Advance only following successful verification.
    retained = RetainedLogCheckpoint(
        size=support["checkpoint"]["size"],
        root=support["checkpoint"]["root"],
    )

assert retained.size == DEVELOPMENT_COUNT

assert retained.root == log_roots[DEVELOPMENT_COUNT].hex()


# ------------------------------------------------------------
# Controlled infrastructure tests
# ------------------------------------------------------------

# Use a non-genesis checkpoint with non-empty proof paths.
test_index = 3

sample = signed_development_evidence[test_index]
support = development_log_supports[test_index]
old_checkpoint = development_log_histories[test_index]

assert support["inclusion_proof"]
assert support["consistency_proof"]


def check_log_test(
    name,
    candidate,
    expected,
    condition=None,
    status=None,
    commitment=sample["commitment"],
    previous=old_checkpoint,
):

    result = verify_log_support(
        commitment,
        sample["evidence"]["sequence"],
        candidate,
        previous,
    )

    assert aggregate_decision(
        result["checks"]
    ) == expected, (name, result)

    if condition is not None:
        assert result["checks"][condition] == status, result

    print(f"{name}: PASS")


print("=== CONTROLLED LOG VERIFICATION TESTS ===")

# 1. Valid checkpoint and proofs.
check_log_test(
    "Valid log evidence",
    support, ACCEPT,
)

# 2. Different evidence commitment.
check_log_test(
    "Incorrect evidence commitment",
    support, REJECT,
    "R03.log_inclusion", FAIL,
    commitment=flip_digest_bit(
        bytes.fromhex(sample["commitment"])
    ).hex(),
)

# 3. Inclusion proof unavailable.
modified = deepcopy(support)
del modified["inclusion_proof"]

check_log_test(
    "Missing inclusion proof",
    modified, INDETERMINATE,
    "R03.log_inclusion", UNKNOWN,
)

# 4. Inclusion proof modified.
modified = deepcopy(support)

modified["inclusion_proof"][0] = flip_digest_bit(
    modified["inclusion_proof"][0]
)

check_log_test(
    "Invalid inclusion proof",
    modified, REJECT,
    "R03.log_inclusion", FAIL,
)

# 5. Invalid operator signature.
modified = deepcopy(support)

modified["operator_signature"] = flip_digest_bit(
    modified["operator_signature"]
)

check_log_test(
    "Invalid operator signature",
    modified, REJECT,
    "R04.log_operator", FAIL,
)

# 6. Missing operator signature.
modified = deepcopy(support)
del modified["operator_signature"]

check_log_test(
    "Missing operator signature",
    modified, INDETERMINATE,
    "R04.log_operator", UNKNOWN,
)

# 7. Insufficient witnesses.
modified = deepcopy(support)
modified["cosignatures"] = modified["cosignatures"][:2]

check_log_test(
    "Insufficient witness quorum",
    modified, INDETERMINATE,
    "R05.witness_quorum", UNKNOWN,
)

# 8. Duplicate signatures cannot create a quorum.
modified = deepcopy(support)

modified["cosignatures"] = [
    modified["cosignatures"][0]
] * WITNESS_QUORUM

check_log_test(
    "Duplicate witness votes",
    modified, INDETERMINATE,
    "R05.witness_quorum", UNKNOWN,
)

# 9. Three valid votes remain sufficient when an
# additional witness provides an invalid signature.
modified = deepcopy(support)

name, signature = modified["cosignatures"][-1]

modified["cosignatures"][-1] = (
    name, flip_digest_bit(signature)
)

check_log_test(
    "Valid quorum with one invalid vote",
    modified, ACCEPT,
    "R05.witness_quorum", PASS,
)

# 10. Missing consistency proof.
modified = deepcopy(support)
del modified["consistency_proof"]

check_log_test(
    "Missing consistency proof",
    modified, INDETERMINATE,
    "R04.log_consistency", UNKNOWN,
)

# 11. Modified consistency proof.
modified = deepcopy(support)

modified["consistency_proof"][0] = flip_digest_bit(
    modified["consistency_proof"][0]
)

check_log_test(
    "Invalid consistency proof",
    modified, REJECT,
    "R04.log_consistency", FAIL,
)

# 12. Previously retained checkpoint unavailable.
check_log_test(
    "Missing retained checkpoint",
    support, INDETERMINATE,
    "R04.log_consistency", UNKNOWN,
    previous=None,
)

# 13. All transparency-log support unavailable.
check_log_test(
    "Unavailable log evidence",
    None, INDETERMINATE,
)

# 14. Authenticated conflicting history.
# Construct a genuine operator-signed alternative root
# at the same size as the retained checkpoint.
alternative_entries = [
    flip_digest_bit(log_entries[0]),
    *log_entries[1:old_checkpoint.size],
]

alternative_checkpoint = make_checkpoint(
    alternative_entries
)

assert alternative_checkpoint["root"] != old_checkpoint.root

alternative_support = {
    "checkpoint": alternative_checkpoint,
    "operator_signature": sign_checkpoint(
        alternative_checkpoint,
        operator_key,
        OPERATOR_DOMAIN,
    ),
    "cosignatures": [],
}

check_log_test(
    "Authenticated conflicting checkpoint",
    alternative_support, REJECT,
    "R04.log_consistency", FAIL,
)

# 15. Malformed checkpoint.
modified = deepcopy(support)
modified["checkpoint"]["root"] = "invalid_digest"

check_log_test(
    "Malformed checkpoint",
    modified, REJECT,
    "R04.log_operator", FAIL,
)


# ------------------------------------------------------------
# Report results
# ------------------------------------------------------------

decisions = [
    aggregate_decision(result["checks"])
    for result in development_log_results
]

print("\n=== DEVELOPMENT LOG VERIFICATION ===")
print("Infrastructure conditions:", len(LOG_CHECKS))
print("Development controls:", len(decisions))
print("Retained checkpoints:", len(development_log_histories))
print("Accepted:", decisions.count(ACCEPT))
print("Rejected:", decisions.count(REJECT))
print("Indeterminate:", decisions.count(INDETERMINATE))

print("\n=== FINAL LOG STATE ===")
print("Tree size:", retained.size)
print("Final root:", retained.root)

print("\n=== IMPLEMENTATION VALIDATION ===")
print("Controlled tests: 15/15 PASS")
print("Inclusion verification: PASS")
print("Operator authentication: PASS")
print("Consistency verification: PASS")
print("Witness quorum verification: PASS")
print("Missing evidence handling: PASS")

print("\nVALIDATION: PASS")

=== CONTROLLED LOG VERIFICATION TESTS ===
Valid log evidence: PASS
Incorrect evidence commitment: PASS
Missing inclusion proof: PASS
Invalid inclusion proof: PASS
Invalid operator signature: PASS
Missing operator signature: PASS
Insufficient witness quorum: PASS
Duplicate witness votes: PASS
Valid quorum with one invalid vote: PASS
Missing consistency proof: PASS
Invalid consistency proof: PASS
Missing retained checkpoint: PASS
Unavailable log evidence: PASS
Authenticated conflicting checkpoint: PASS
Malformed checkpoint: PASS

=== DEVELOPMENT LOG VERIFICATION ===
Infrastructure conditions: 4
Development controls: 65
Retained checkpoints: 65
Accepted: 65
Rejected: 0
Indeterminate: 0

=== FINAL LOG STATE ===
Tree size: 65
Final root: b4aacc347149986abf4f54694aa5877c740931b34d57701ca31e9c701b7a14e3

=== IMPLEMENTATION VALIDATION ===
Controlled tests: 15/15 PASS
Inclusion verification: PASS
Operator authentication: PASS
Consistency verification: PASS
Witness quorum verification: PASS
Miss

In [15]:
# ============================================================
# CELL 13 — Ethereum infrastructure verification
# ============================================================
# Verify actual local Ethereum transactions, anchor events
# and contract storage.
#
# References:
# REQUIREMENTS.md, R03-R05.
# EXPERIMENTAL_DESIGN.md, Section 5.
# https://pypi.org/project/eth-tester/0.14.0b1/
#
# The configured local backend is a trust dependency.
# Public-network consensus and finality are not verified.

from eth_tester.exceptions import TransactionNotFound, BlockNotFound

ETHEREUM_CHECKS = (
    "R03.transaction_inclusion",
    "R03.anchor_execution",
    "R04.anchor_event",
    "R04.contract_storage",
)

ANCHOR_EVENT_TOPIC = "0x" + keccak(
    b"Anchored(uint256,bytes32,bytes32)"
).hex()


def valid_transaction_hash(value):
    """Validate the representation of an Ethereum transaction hash."""

    return (
        type(value) is str
        and re.fullmatch(r"0x[0-9a-fA-F]{64}", value)
        is not None
    )


def verify_ethereum_support(
    evidence,
    computed_commitment,
    support,
    anchor,
):
    """Verify anchoring using the configured local Ethereum backend."""

    checks = {
        name: UNKNOWN
        for name in ETHEREUM_CHECKS
    }

    sequence = None

    if type(evidence) is dict:

        value = evidence.get("sequence")

        if type(value) is int and value >= 0:
            sequence = value

    # --------------------------------------------------------
    # R04: Independently query configured contract storage
    # --------------------------------------------------------

    if (
        anchor is not None
        and sequence is not None
        and valid_digest(computed_commitment)
    ):

        stored_commitment = anchor.commitment_at(sequence)

        # An absent anchor does not prove a conflicting anchor.
        if stored_commitment is not None:

            checks["R04.contract_storage"] = check_status(
                stored_commitment.hex() == computed_commitment
            )

    # --------------------------------------------------------
    # Validate submitted transaction reference
    # --------------------------------------------------------

    if support is None:
        return {"checks": checks}

    if type(support) is not dict:

        checks["R03.transaction_inclusion"] = FAIL
        return {"checks": checks}

    tx_hash = support.get("transaction_hash")

    if tx_hash is None:
        return {"checks": checks}

    if not valid_transaction_hash(tx_hash):

        checks["R03.transaction_inclusion"] = FAIL
        return {"checks": checks}

    if anchor is None:
        return {"checks": checks}

    tx_hash = tx_hash.lower()

    # --------------------------------------------------------
    # Retrieve transaction and receipt from the trusted backend
    # --------------------------------------------------------

    try:

        transaction = anchor.tester.get_transaction_by_hash(
            tx_hash
        )

        receipt = anchor.tester.get_transaction_receipt(
            tx_hash
        )

    except TransactionNotFound:

        # Unknown transaction is insufficient evidence.
        return {"checks": checks}

    if (
        receipt["block_number"] is None
        or receipt["block_hash"] is None
        or receipt["transaction_index"] is None
    ):
        return {"checks": checks}

    # --------------------------------------------------------
    # R03: Verify local block membership
    # --------------------------------------------------------

    try:

        block = anchor.tester.get_block_by_number(
            receipt["block_number"],
            full_transactions=False,
        )

    except BlockNotFound:
        return {"checks": checks}

    tx_index = receipt["transaction_index"]

    membership_valid = (
        transaction["hash"].lower() == tx_hash
        and receipt["transaction_hash"].lower() == tx_hash
        and block["hash"] == receipt["block_hash"]
        and block["number"] == receipt["block_number"]
        and transaction["block_hash"] == receipt["block_hash"]
        and transaction["block_number"] == receipt["block_number"]
        and transaction["transaction_index"] == tx_index
        and 0 <= tx_index < len(block["transactions"])
        and block["transactions"][tx_index].lower() == tx_hash
    )

    checks["R03.transaction_inclusion"] = check_status(
        membership_valid
    )

    if not membership_valid:
        return {"checks": checks}

    # --------------------------------------------------------
    # R03: Verify successful execution and configured identity
    # --------------------------------------------------------

    destination = transaction.get("to")
    sender = transaction.get("from")

    execution_valid = (
        receipt["status"] == 1
        and type(destination) is str
        and destination.lower() == anchor.address.lower()
        and type(sender) is str
        and sender.lower() == anchor.relay.lower()
    )

    checks["R03.anchor_execution"] = check_status(
        execution_valid
    )

    if not execution_valid:
        return {"checks": checks}

    # --------------------------------------------------------
    # R04: Inspect the actual anchor event
    # --------------------------------------------------------

    matching_events = [
        event for event in receipt["logs"]
        if (
            event["address"].lower() == anchor.address.lower()
            and event["topics"]
            and event["topics"][0] == ANCHOR_EVENT_TOPIC
        )
    ]

    if len(matching_events) != 1:

        checks["R04.anchor_event"] = FAIL
        return {"checks": checks}

    event = matching_events[0]

    if (
        len(event["topics"]) != 2
        or event["transaction_hash"].lower() != tx_hash
        or event["block_hash"] != receipt["block_hash"]
        or event["block_number"] != receipt["block_number"]
        or event["transaction_index"] != tx_index
    ):

        checks["R04.anchor_event"] = FAIL
        return {"checks": checks}

    event_data = bytes.fromhex(event["data"][2:])

    if len(event_data) != 64:

        checks["R04.anchor_event"] = FAIL
        return {"checks": checks}

    event_sequence = int(event["topics"][1], 16)

    event_commitment, event_previous = decode(
        ["bytes32", "bytes32"],
        event_data,
    )

    comparisons = []

    if sequence is not None:
        comparisons.append(event_sequence == sequence)

    if valid_digest(computed_commitment):

        comparisons.append(
            event_commitment.hex() == computed_commitment
        )

    if (
        type(evidence) is dict
        and "previous_commitment" in evidence
    ):

        previous = evidence["previous_commitment"]

        if previous is None:

            comparisons.append(
                event_previous == ZERO_COMMITMENT
            )

        elif valid_digest(previous):

            comparisons.append(
                event_previous.hex() == previous
            )

        else:
            comparisons.append(False)

    if False in comparisons:

        checks["R04.anchor_event"] = FAIL

    elif len(comparisons) == 3:

        checks["R04.anchor_event"] = PASS

    return {"checks": checks}


# ------------------------------------------------------------
# Validate all development anchors
# ------------------------------------------------------------

development_ethereum_supports = {}
development_ethereum_results = []

for item, receipt in zip(
    signed_development_evidence,
    ethereum_receipts,
    strict=True,
):

    evidence = item["evidence"]
    sequence = evidence["sequence"]

    support = {
        "transaction_hash": receipt["transaction_hash"]
    }

    common = verify_common_crypto(
        item,
        development_histories[sequence],
    )

    result = verify_ethereum_support(
        evidence,
        common["computed_commitment"],
        support,
        ethereum_anchor,
    )

    assert aggregate_decision(result["checks"]) == ACCEPT, result

    development_ethereum_supports[sequence] = support
    development_ethereum_results.append(result)


# ------------------------------------------------------------
# Controlled infrastructure tests
# ------------------------------------------------------------

sample = signed_development_evidence[1]

test_evidence = sample["evidence"]
test_commitment = sample["commitment"]
test_support = development_ethereum_supports[1]

state_before = anchor_state(ethereum_anchor)


def check_ethereum_test(
    name,
    expected,
    condition=None,
    status=None,
    evidence=test_evidence,
    commitment=test_commitment,
    support=test_support,
    anchor=ethereum_anchor,
):

    result = verify_ethereum_support(
        evidence,
        commitment,
        support,
        anchor,
    )

    assert aggregate_decision(
        result["checks"]
    ) == expected, (name, result)

    if condition is not None:
        assert result["checks"][condition] == status, result

    print(f"{name}: PASS")


print("=== CONTROLLED ETHEREUM VERIFICATION TESTS ===")

# 1. Valid transaction and stored commitment.
check_ethereum_test(
    "Valid blockchain evidence",
    ACCEPT,
)

# 2. Submitted evidence commitment differs from the anchor.
check_ethereum_test(
    "Incorrect evidence commitment",
    REJECT,
    "R04.contract_storage",
    FAIL,
    commitment=flip_digest_bit(
        bytes.fromhex(test_commitment)
    ).hex(),
)

# 3. Submitted predecessor differs from the anchor event.
modified = deepcopy(test_evidence)

modified["previous_commitment"] = flip_digest_bit(
    bytes.fromhex(modified["previous_commitment"])
).hex()

check_ethereum_test(
    "Incorrect predecessor",
    REJECT,
    "R04.anchor_event",
    FAIL,
    evidence=modified,
    commitment=evidence_commitment(modified),
)

# 4. A genuine anchor transaction belongs to another batch.
check_ethereum_test(
    "Incorrect transaction reference",
    REJECT,
    "R04.anchor_event",
    FAIL,
    support=development_ethereum_supports[0],
)

# 5. Transaction reference unavailable.
check_ethereum_test(
    "Missing transaction reference",
    INDETERMINATE,
    "R03.transaction_inclusion",
    UNKNOWN,
    support=None,
)

# 6. Well-formed but unknown transaction hash.
unknown_hash = "0x" + hashlib.sha256(
    b"unknown-development-transaction"
).hexdigest()

check_ethereum_test(
    "Unknown transaction hash",
    INDETERMINATE,
    "R03.transaction_inclusion",
    UNKNOWN,
    support={"transaction_hash": unknown_hash},
)

# 7. Malformed transaction hash.
check_ethereum_test(
    "Malformed transaction reference",
    REJECT,
    "R03.transaction_inclusion",
    FAIL,
    support={"transaction_hash": "invalid_hash"},
)

# 8. Configured backend unavailable.
check_ethereum_test(
    "Unavailable blockchain backend",
    INDETERMINATE,
    "R03.transaction_inclusion",
    UNKNOWN,
    anchor=None,
)

# 9. Entire evidence payload unavailable.
check_ethereum_test(
    "Unavailable evidence payload",
    INDETERMINATE,
    "R04.anchor_event",
    UNKNOWN,
    evidence=None,
    commitment=None,
)

# 10. Successful transaction that deployed the contract
# but did not anchor a measurement batch.
check_ethereum_test(
    "Non-anchoring transaction",
    REJECT,
    "R03.anchor_execution",
    FAIL,
    support={
        "transaction_hash":
            ethereum_anchor.deployment_receipt["transaction_hash"]
    },
)

# 11. Actual reverted transaction in an isolated backend.
isolated_anchor = EthereumAnchor()

failed_receipt = isolated_anchor.append(
    sequence=1,
    commitment=bytes.fromhex(test_commitment),
    previous=ZERO_COMMITMENT,
)

assert failed_receipt["status"] == 0

check_ethereum_test(
    "Reverted anchoring transaction",
    REJECT,
    "R03.anchor_execution",
    FAIL,
    support={
        "transaction_hash": failed_receipt["transaction_hash"]
    },
    anchor=isolated_anchor,
)

# 12. No commitment has been recorded in the isolated contract.
assert isolated_anchor.count() == 0

check_ethereum_test(
    "Unanchored evidence",
    INDETERMINATE,
    "R04.contract_storage",
    UNKNOWN,
    support=None,
    anchor=isolated_anchor,
)

# Verification must not modify the original development chain.
assert anchor_state(ethereum_anchor) == state_before


# ------------------------------------------------------------
# Report results
# ------------------------------------------------------------

decisions = [
    aggregate_decision(result["checks"])
    for result in development_ethereum_results
]

print("\n=== DEVELOPMENT ETHEREUM VERIFICATION ===")
print("Infrastructure conditions:", len(ETHEREUM_CHECKS))
print("Development controls:", len(decisions))
print("Transaction references:", len(development_ethereum_supports))
print("Accepted:", decisions.count(ACCEPT))
print("Rejected:", decisions.count(REJECT))
print("Indeterminate:", decisions.count(INDETERMINATE))

print("\n=== FINAL ETHEREUM STATE ===")
print("Stored commitments:", ethereum_anchor.count())
print("Final commitment:", ethereum_anchor.head().hex())

print("\n=== IMPLEMENTATION VALIDATION ===")
print("Controlled tests: 12/12 PASS")
print("Transaction inclusion: PASS")
print("Anchor execution verification: PASS")
print("Anchor event verification: PASS")
print("Contract storage verification: PASS")
print("Development contract state preservation: PASS")
print("Public-network consensus: NOT EVALUATED")
print("Public-network finality: NOT EVALUATED")

print("\nVALIDATION: PASS")

=== CONTROLLED ETHEREUM VERIFICATION TESTS ===
Valid blockchain evidence: PASS
Incorrect evidence commitment: PASS
Incorrect predecessor: PASS
Incorrect transaction reference: PASS
Missing transaction reference: PASS
Unknown transaction hash: PASS
Malformed transaction reference: PASS
Unavailable blockchain backend: PASS
Unavailable evidence payload: PASS
Non-anchoring transaction: PASS
Reverted anchoring transaction: PASS
Unanchored evidence: PASS

=== DEVELOPMENT ETHEREUM VERIFICATION ===
Infrastructure conditions: 4
Development controls: 65
Transaction references: 65
Accepted: 65
Rejected: 0
Indeterminate: 0

=== FINAL ETHEREUM STATE ===
Stored commitments: 65
Final commitment: b95e58c148f97265dfef56a9fd718d3e38e357a458d629112d443a6f1638f2ec

=== IMPLEMENTATION VALIDATION ===
Controlled tests: 12/12 PASS
Transaction inclusion: PASS
Anchor execution verification: PASS
Anchor event verification: PASS
Contract storage verification: PASS
Development contract state preservation: PASS
Pub

In [16]:
# ============================================================
# CELL 14 — Six-architecture decision integration
# ============================================================
# Combine previously validated verification components.
#
# References:
# EXPERIMENTAL_DESIGN.md, Sections 3 and 7.
# PROTOCOL_ADDENDUM.md, Sections 1-4.
# PROTOCOL_AMENDMENT_02.md.
#
# Verification must not receive hidden scenario labels
# or expected reference outcomes.

ARCHITECTURES = {
    "A1": ("signatures", False),
    "A2": ("signatures", True),
    "B1": ("transparency_log", False),
    "B2": ("transparency_log", True),
    "C1": ("ethereum", False),
    "C2": ("ethereum", True),
}


def verify_architecture(
    architecture,
    envelope,
    history,
    *,
    policy_context=None,
    certificate=None,
    log_support=None,
    log_history=None,
    ethereum_support=None,
    anchor=None,
):
    """Evaluate one architecture using available evidence."""

    infrastructure, use_policy = ARCHITECTURES[architecture]

    # Common cryptographic verification.
    common = verify_common_crypto(envelope, history)

    layers = {
        "common": common["checks"]
    }

    commitment = common["computed_commitment"]

    evidence = (
        envelope.get("evidence")
        if type(envelope) is dict else None
    )

    public_key = (
        envelope.get("public_key")
        if type(envelope) is dict else None
    )

    sequence = (
        evidence.get("sequence")
        if type(evidence) is dict else None
    )

    # Infrastructure-specific verification.
    if infrastructure == "transparency_log":

        result = verify_log_support(
            commitment,
            sequence,
            log_support,
            log_history,
        )

        layers["infrastructure"] = result["checks"]

    elif infrastructure == "ethereum":

        result = verify_ethereum_support(
            evidence,
            commitment,
            ethereum_support,
            anchor,
        )

        layers["infrastructure"] = result["checks"]

    # Identical additional evidence rules for A2, B2 and C2.
    if use_policy:

        result = evaluate_evidence_policy(
            evidence,
            public_key,
            certificate,
            policy_context,
        )

        layers["policy"] = result["checks"]

    # Combine mandatory conditions without overwriting checks.
    checks = {}

    for layer_checks in layers.values():

        assert not set(checks).intersection(layer_checks)

        checks.update(layer_checks)

    return {
        "architecture": architecture,
        "decision": aggregate_decision(checks),
        "checks": checks,
        "layer_decisions": {
            name: aggregate_decision(layer_checks)
            for name, layer_checks in layers.items()
        },
        "violations": [
            name for name, status in checks.items()
            if status == FAIL
        ],
        "unresolved": [
            name for name, status in checks.items()
            if status == UNKNOWN
        ],
    }


# ------------------------------------------------------------
# Construct equivalent development verification inputs
# ------------------------------------------------------------

def development_case(sequence):
    """Provide available evidence for a development control."""

    return {
        "envelope": signed_development_evidence[sequence],
        "history": development_histories[sequence],
        "policy_context": development_policy_contexts[sequence],
        "certificate": development_certificates[sequence],
        "log_support": development_log_supports[sequence],
        "log_history": development_log_histories[sequence],
        "ethereum_support": development_ethereum_supports[sequence],
        "anchor": ethereum_anchor,
    }


def verify_all_architectures(case):
    """Apply all six architectures to equivalent inputs."""

    return {
        architecture: verify_architecture(
            architecture, **case
        )
        for architecture in ARCHITECTURES
    }


# Preserve development contract state for read-only validation.
integration_state_before = anchor_state(ethereum_anchor)

integrated_development_results = []

for sequence in range(DEVELOPMENT_COUNT):

    results = verify_all_architectures(
        development_case(sequence)
    )

    for architecture, result in results.items():

        assert result["decision"] == ACCEPT, result
        assert not result["violations"]
        assert not result["unresolved"]

        infrastructure, use_policy = ARCHITECTURES[architecture]

        expected_conditions = len(COMMON_CHECKS)

        if infrastructure == "transparency_log":
            expected_conditions += len(LOG_CHECKS)

        elif infrastructure == "ethereum":
            expected_conditions += len(ETHEREUM_CHECKS)

        if use_policy:
            expected_conditions += len(POLICY_CHECKS)
            expected_conditions += len(CERTIFICATE_CHECKS)

        assert len(result["checks"]) == expected_conditions

        integrated_development_results.append({
            "sequence": sequence,
            **result,
        })


# ------------------------------------------------------------
# Controlled integration tests
# ------------------------------------------------------------

def check_architecture_test(name, case, expected):
    """Compare actual decisions with explicit test expectations."""

    results = verify_all_architectures(case)

    actual = {
        architecture: result["decision"]
        for architecture, result in results.items()
    }

    assert actual == expected, (
        name, expected, actual
    )

    print(f"{name}: PASS")

    return results


all_accept = dict.fromkeys(ARCHITECTURES, ACCEPT)
all_reject = dict.fromkeys(ARCHITECTURES, REJECT)
all_indeterminate = dict.fromkeys(ARCHITECTURES, INDETERMINATE)

policy_indeterminate = {
    **all_accept,
    "A2": INDETERMINATE,
    "B2": INDETERMINATE,
    "C2": INDETERMINATE,
}

policy_reject = {
    **all_accept,
    "A2": REJECT,
    "B2": REJECT,
    "C2": REJECT,
}

log_indeterminate = {
    **all_accept,
    "B1": INDETERMINATE,
    "B2": INDETERMINATE,
}

ethereum_indeterminate = {
    **all_accept,
    "C1": INDETERMINATE,
    "C2": INDETERMINATE,
}

print("=== CONTROLLED INTEGRATION TESTS ===")

# 1. Post-signing measurement modification.
case = development_case(3)
case["envelope"] = deepcopy(case["envelope"])
case["envelope"]["evidence"]["records"][0]["voltage"] = "999"

check_architecture_test(
    "Post-signing modification",
    case,
    all_reject,
)

# 2. Missing institutional certificate.
case = development_case(3)
case["certificate"] = None

check_architecture_test(
    "Missing certificate",
    case,
    policy_indeterminate,
)

# 3. Missing common retained history.
case = development_case(3)
case["history"] = None

check_architecture_test(
    "Missing common history",
    case,
    all_indeterminate,
)

# 4. Missing transparency-log evidence.
case = development_case(3)
case["log_support"] = None

check_architecture_test(
    "Missing log evidence",
    case,
    log_indeterminate,
)

# 5. Missing Ethereum transaction reference.
case = development_case(3)
case["ethereum_support"] = None

check_architecture_test(
    "Missing Ethereum evidence",
    case,
    ethereum_indeterminate,
)

# 6. Insufficient witness quorum.
case = development_case(3)
case["log_support"] = deepcopy(case["log_support"])
case["log_support"]["cosignatures"] = (
    case["log_support"]["cosignatures"][:2]
)

check_architecture_test(
    "Insufficient witness quorum",
    case,
    log_indeterminate,
)

# 7. Entire submitted evidence envelope unavailable.
case = development_case(3)
case["envelope"] = None

check_architecture_test(
    "Unavailable evidence payload",
    case,
    all_indeterminate,
)


# ------------------------------------------------------------
# Correctly signed AND committed semantic failure
# ------------------------------------------------------------
# Create an independent experimental history.
# Do not reuse anchors from the unmodified evidence.

semantic_case = development_case(0)

modified = deepcopy(semantic_case["envelope"])

modified["evidence"]["claims"][
    "measurement_units"
]["current"] = "mA"

semantic_case["envelope"] = resign_envelope(
    modified,
    source_private_keys[
        modified["evidence"]["source_id"]
    ],
)

modified_commitment = bytes.fromhex(
    semantic_case["envelope"]["commitment"]
)

# Create a valid witnessed log for the modified evidence.
semantic_entries = [modified_commitment]
semantic_checkpoint = make_checkpoint(semantic_entries)

semantic_operator_signature = sign_checkpoint(
    semantic_checkpoint,
    operator_key,
    OPERATOR_DOMAIN,
)

semantic_witnesses = {
    name: Witness(name)
    for name in witness_keys
}

semantic_cosignatures = [
    witness.cosign(
        semantic_checkpoint,
        semantic_operator_signature,
        [],
    )
    for witness in semantic_witnesses.values()
]

assert all(
    vote is not None
    for vote in semantic_cosignatures
)

semantic_bundle = {
    "checkpoint": semantic_checkpoint,
    "operator_signature": semantic_operator_signature,
    "cosignatures": semantic_cosignatures,
}

semantic_case["log_support"] = build_log_support(
    semantic_entries,
    0,
    semantic_bundle,
    0,
)

# Independently anchor the same modified commitment.
semantic_anchor = EthereumAnchor()

semantic_receipt = semantic_anchor.append(
    0,
    modified_commitment,
    ZERO_COMMITMENT,
)

assert semantic_receipt["status"] == 1

semantic_case["anchor"] = semantic_anchor

semantic_case["ethereum_support"] = {
    "transaction_hash": semantic_receipt["transaction_hash"]
}

# 8. Incorrect unit committed consistently across infrastructures.
semantic_results = check_architecture_test(
    "Correctly committed incorrect unit",
    semantic_case,
    policy_reject,
)

# Confirm that rejection comes from the evidence policy,
# not a cryptographic or infrastructure inconsistency.
for architecture, result in semantic_results.items():

    assert result["layer_decisions"]["common"] == ACCEPT

    if architecture.startswith(("B", "C")):
        assert result["layer_decisions"]["infrastructure"] == ACCEPT

    if architecture.endswith("2"):
        assert "R08.units" in result["violations"]

# 9. Incorrect unit combined with a missing certificate.
combined_case = dict(semantic_case)
combined_case["certificate"] = None

check_architecture_test(
    "Semantic violation with missing certificate",
    combined_case,
    policy_reject,
)

# 10. Invalid signature combined with a missing certificate.
case = development_case(3)
case["envelope"] = deepcopy(case["envelope"])

case["envelope"]["signature"] = flip_digest_bit(
    case["envelope"]["signature"]
)

case["certificate"] = None

check_architecture_test(
    "Cryptographic violation with missing certificate",
    case,
    all_reject,
)


# ------------------------------------------------------------
# Final validation and reporting
# ------------------------------------------------------------

assert anchor_state(ethereum_anchor) == integration_state_before

assert len(integrated_development_results) == (
    DEVELOPMENT_COUNT * len(ARCHITECTURES)
)

print("\n=== SIX-ARCHITECTURE DEVELOPMENT VALIDATION ===")

for architecture in ARCHITECTURES:

    results = [
        result for result in integrated_development_results
        if result["architecture"] == architecture
    ]

    decisions = [
        result["decision"] for result in results
    ]

    print(
        f"{architecture}: "
        f"conditions={len(results[0]['checks'])}, "
        f"accepted={decisions.count(ACCEPT)}, "
        f"rejected={decisions.count(REJECT)}, "
        f"indeterminate={decisions.count(INDETERMINATE)}"
    )

print("\n=== CORRECTLY COMMITTED INCORRECT UNIT ===")

for architecture, result in semantic_results.items():
    print(f"{architecture}: {result['decision']}")

print("\n=== IMPLEMENTATION VALIDATION ===")
print("Development architecture decisions:",
      len(integrated_development_results))

print("Controlled integration tests: 10/10 PASS")
print("Controlled architecture decisions: 60")
print("Common commitment equivalence: PASS")
print("Original development contract state: PRESERVED")
print("Evaluation scenarios executed: 0")

print("\nVALIDATION: PASS")

=== CONTROLLED INTEGRATION TESTS ===
Post-signing modification: PASS
Missing certificate: PASS
Missing common history: PASS
Missing log evidence: PASS
Missing Ethereum evidence: PASS
Insufficient witness quorum: PASS
Unavailable evidence payload: PASS
Correctly committed incorrect unit: PASS
Semantic violation with missing certificate: PASS
Cryptographic violation with missing certificate: PASS

=== SIX-ARCHITECTURE DEVELOPMENT VALIDATION ===
A1: conditions=5, accepted=65, rejected=0, indeterminate=0
A2: conditions=23, accepted=65, rejected=0, indeterminate=0
B1: conditions=9, accepted=65, rejected=0, indeterminate=0
B2: conditions=27, accepted=65, rejected=0, indeterminate=0
C1: conditions=9, accepted=65, rejected=0, indeterminate=0
C2: conditions=27, accepted=65, rejected=0, indeterminate=0

=== CORRECTLY COMMITTED INCORRECT UNIT ===
A1: ACCEPT
A2: REJECT
B1: ACCEPT
B2: REJECT
C1: ACCEPT
C2: REJECT

=== IMPLEMENTATION VALIDATION ===
Development architecture decisions: 390
Controlled 